In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def kaiming_init(m):
    if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d, nn.Linear)):
        nn.init.kaiming_uniform_(m.weight, a=0.2)
        if m.bias is not None: nn.init.zeros_(m.bias)
            
class VAE(nn.Module):
    def __init__(self, nfeat=1000, ncode=10, ch1=32, ch2=64, ch3 = 128, ch4 = 256, dropout=0.02, use_bn=True):
        super().__init__()
        self.nfeat = int(nfeat)
        self.ncode = int(ncode)
        self.ch1, self.ch2, self.ch3, self.ch4 = ch1, ch2, ch3, ch4

        Norm = nn.BatchNorm1d if use_bn else nn.Identity
        act  = nn.LeakyReLU(0.2, inplace=True)

        # Encoder
        '''
        self.enc = nn.Sequential(
            nn.Conv1d(1, ch1, kernel_size=9, stride=1, padding=4),
            Norm(ch1), act,
            nn.Conv1d(ch1, ch2, kernel_size=7, stride=1, padding=3), 
            Norm(ch2), act
        )
        '''
        self.enc = nn.Sequential( 
            nn.Conv1d(1, ch1, kernel_size=9, stride=1, padding=4), # 1000 -> 1000 
            Norm(ch1), act, 
            
            nn.Conv1d(ch1, ch2, kernel_size=7, stride=2, padding=3), # 1000 -> 500 
            Norm(ch2), act, 
            
            nn.Conv1d(ch2, ch3, kernel_size=5, stride=2, padding=2), # 500 -> 250 
            Norm(ch3), act, 
            
            nn.Conv1d(ch3, ch4, kernel_size=3, stride=2, padding=1), # 250 -> 125 
            Norm(ch4), act, 
        )

        '''
        with torch.no_grad():
            dummy = torch.zeros(1, 1, self.nfeat)
            h = self.enc(dummy)                   
            _, _, L_enc = h.shape
        self.L_enc = L_enc

        self.enc_ln = nn.LayerNorm([ch2, self.L_enc])
        self.flat   = nn.Flatten()
        enc_dim     = ch2 * self.L_enc

        self.mu = nn.Linear(enc_dim, ncode)
        self.lv = nn.Linear(enc_dim, ncode)
        '''
        with torch.no_grad():
            dummy = torch.zeros(1, 1, self.nfeat)
            h = self.enc(dummy)
            _, _, L_enc = h.shape
        
        self.L_enc = L_enc
        enc_dim = ch4 * self.L_enc
        
        self.enc_ln = nn.LayerNorm([ch4, self.L_enc])
        self.flat   = nn.Flatten()
        
        self.mu = nn.Linear(enc_dim, ncode)
        self.lv = nn.Linear(enc_dim, ncode)


        # Decoder
        self.fc = nn.Linear(ncode, enc_dim)
        
        self.dec_preln = nn.LayerNorm([ch4, self.L_enc])

        self.dec = nn.Sequential(
            nn.ConvTranspose1d(ch4, ch3, kernel_size=3, stride=2, padding=1, output_padding=1),
            act,
        
            nn.ConvTranspose1d(ch3, ch2, kernel_size=5, stride=2, padding=2, output_padding=1),
            act,
        
            nn.ConvTranspose1d(ch2, ch1, kernel_size=7, stride=2, padding=3, output_padding=1),
            act,
        
            nn.Conv1d(ch1, 1, kernel_size=9, padding=4),
        )
        
        self.apply(kaiming_init)
        
    def encode(self, x):
        h = self.enc(x)             
        h = self.enc_ln(h)
        h = self.flat(h)
        mu = self.mu(h)
        lv = torch.clamp(self.lv(h), -8.0, 8.0)
        return mu, lv

    def reparameterize(self, mu, lv):
        std = torch.exp(0.5*lv)
        eps = torch.randn_like(std)
        return mu + std*eps

    def decode(self, z):
        h = self.fc(z).view(z.size(0), self.ch4, self.L_enc)
        h = self.dec_preln(h)
        xhat = self.dec(h)
        #return self.crop(xhat)
        return xhat


    def forward(self, x):
        if x.dim()==2: x = x.unsqueeze(1)
        mu, lv = self.encode(x)
        #z = mu
        z = self.reparameterize(mu, lv)
        xhat = self.decode(z)
        return xhat, mu, lv, z

    def compute_kernel(self, x, y):
        x_size = x.size(0)
        y_size = y.size(0)
        dim = x.size(1)
        x = x.unsqueeze(1)        
        y = y.unsqueeze(0)           
        tiled_x = x.expand(x_size, y_size, dim)
        tiled_y = y.expand(x_size, y_size, dim)
        kernel_input = (tiled_x - tiled_y).pow(2).mean(2)
        return torch.exp(-kernel_input)

    def compute_mmd(self, x, y):
        xx = self.compute_kernel(x, x)
        yy = self.compute_kernel(y, y)
        xy = self.compute_kernel(x, y)
        return torch.mean(xx) + torch.mean(yy) - 2*torch.mean(xy)
    
        
    def loss(self, x, w=None, alpha=0.0, lam=0.1, beta=1.0):
        if x.dim() == 2:
            x = x.unsqueeze(1)   # (N,1,L)
    
        # weights
        if w is None:
            w = torch.ones_like(x)
        elif w.dim() == 2:
            w = w.unsqueeze(1)
        w = w.clamp_min(0.0)
    
        # forward
        xhat, mu, lv, z = self.forward(x)   # lv = logvar

        w = torch.sqrt(w)
    
        mse  = torch.sum(0.5 * w * (x - xhat).pow(2))

        #mse = torch.sum(0.5 * w * (x - xhat).pow(2), dim=1) / x.shape[1]
    
        kld = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp())

        #prior = torch.randn_like(z)              
        #mmd = self.compute_mmd(prior, z) * x.size(0) * self.ncode

        l1_loss = (xhat - x).abs()
        mae = (w * l1_loss).sum() / (w.sum().clamp_min(1e-12))

        #total loss
        loss = mse + beta * kld
        return loss, mse, mae, kld


KeyboardInterrupt



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VAE(
    nfeat=1000,
    ncode=16,
    ch1=32,
    ch2=64,
    ch3=128,
    ch4=256,
    dropout=0.02,
    use_bn=True,
).to(device)

In [ ]:
ckpt_path = "/global/cfs/cdirs/m4237/zixinq/data/dr2_catalog/galaxy_spectra_preprocess/vae_4layer_run3/ckpt_epoch010_step0460000.pt"

ckpt = torch.load(ckpt_path, map_location=device)

model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval() 

In [ ]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader, IterableDataset, get_worker_info


def _split_bucket(targetid):
    value = np.asarray(targetid, dtype=np.uint64).copy()
    value ^= value >> np.uint64(30)
    value *= np.uint64(0xBF58476D1CE4E5B9)
    value ^= value >> np.uint64(27)
    value *= np.uint64(0x94D049BB133111EB)
    value ^= value >> np.uint64(31)
    return value % np.uint64(10_000)


def _split_mask(targetid, split):
    if split is None or split == "all":
        return np.ones(len(targetid), dtype=bool)
    bucket = _split_bucket(targetid)
    if split == "train":
        return bucket < 8_000
    if split in {"val", "valid", "validation"}:
        return (bucket >= 8_000) & (bucket < 9_000)
    if split == "test":
        return bucket >= 9_000
    raise ValueError("split must be train, val, test, all, or None")


def _distributed_worker_partition():
    worker = get_worker_info()
    local_id = 0 if worker is None else worker.id
    local_count = 1 if worker is None else worker.num_workers

    if torch.distributed.is_available() and torch.distributed.is_initialized():
        rank = torch.distributed.get_rank()
        world_size = torch.distributed.get_world_size()
    else:
        rank, world_size = 0, 1

    return rank * local_count + local_id, world_size * local_count


class LoaH5Iterable(IterableDataset):
    def __init__(
        self,
        root,
        split="train",
        seed=42,
        shuffle=True,
        include_weights=True,
        min_valid_fraction=0.0,
        files=None,
    ):
        super().__init__()
        self.root = Path(root)
        if files is None:
            if not self.root.is_dir():
                raise FileNotFoundError(f"Dataset directory not found: {self.root}")
            files = sorted(self.root.rglob("*.h5"))
        self.files = [Path(path) for path in files]
        if not self.files:
            raise FileNotFoundError(f"No HDF5 files found under {self.root}")
        if not 0.0 <= min_valid_fraction <= 1.0:
            raise ValueError("min_valid_fraction must be between 0 and 1")
        _split_mask(np.array([], dtype=np.int64), split)
        self.split = split
        self.seed = int(seed)
        self.shuffle = bool(shuffle)
        self.include_weights = bool(include_weights)
        self.min_valid_fraction = float(min_valid_fraction)
        self._iteration = 0

    def __iter__(self):
        global_worker_id, global_worker_count = _distributed_worker_partition()
        iteration = self._iteration
        self._iteration += 1
        rng = np.random.default_rng(
            self.seed + 1_000_003 * iteration + 10_007 * global_worker_id
        )

        files = self.files[global_worker_id::global_worker_count].copy()
        if self.shuffle:
            rng.shuffle(files)

        for path in files:
            with h5py.File(path, "r") as handle:
                if not bool(handle.attrs.get("complete", False)):
                    continue

                status = handle["status"][:]
                targetid = handle["targetid"][:].astype(np.int64, copy=False)
                valid_fraction = handle["valid_frac"][:]
                keep = (status == 0) & _split_mask(targetid, self.split)
                if self.min_valid_fraction > 0:
                    keep &= valid_fraction >= self.min_valid_fraction
                rows = np.flatnonzero(keep)
                if not len(rows):
                    continue

                x = handle["X"][rows]
                mask = handle["M"][rows].astype(bool)
                weight = handle["W"][rows] if self.include_weights else None
                redshift = handle["redshift/z"][rows]
                raw_spectype = handle["redshift/spectype"][rows]
                if raw_spectype.dtype.kind == "S":
                    spectype = np.char.decode(raw_spectype, "utf-8")
                else:
                    spectype = raw_spectype.astype(str)
                norm = handle["norm"][rows]
                offset = handle["offset"][rows]

            order = rng.permutation(len(rows)) if self.shuffle else np.arange(len(rows))
            for index in order:
                item = {
                    "x": torch.from_numpy(x[index]),
                    "mask": torch.from_numpy(mask[index]),
                    "targetid": int(targetid[rows[index]]),
                    "z": float(redshift[index]),
                    "spectype": str(spectype[index]).strip(),
                    "valid_fraction": float(valid_fraction[rows[index]]),
                    "norm": float(norm[index]),
                    "offset": float(offset[index]),
                }
                item["weight"] = (
                    torch.from_numpy(weight[index]) if weight is not None else None
                )
                yield item


def collate_loa(batch):
    weights = None
    if batch[0]["weight"] is not None:
        weights = torch.stack([item["weight"] for item in batch]).unsqueeze(1)
    return {
        "x": torch.stack([item["x"] for item in batch]).unsqueeze(1),
        "weight": weights,
        "mask": torch.stack([item["mask"] for item in batch]).unsqueeze(1),
        "targetid": torch.tensor([item["targetid"] for item in batch], dtype=torch.int64),
        "z": torch.tensor([item["z"] for item in batch], dtype=torch.float32),
        "spectype": [item["spectype"] for item in batch],
        "valid_fraction": torch.tensor(
            [item["valid_fraction"] for item in batch], dtype=torch.float32
        ),
        "norm": torch.tensor([item["norm"] for item in batch], dtype=torch.float64),
        "offset": torch.tensor([item["offset"] for item in batch], dtype=torch.float64),
    }


def make_loader(
    root,
    split="train",
    batch_size=256,
    num_workers=4,
    seed=42,
    shuffle=True,
    include_weights=True,
    min_valid_fraction=0.0,
    drop_last=None,
):
    dataset = LoaH5Iterable(
        root=root,
        split=split,
        seed=seed,
        shuffle=shuffle,
        include_weights=include_weights,
        min_valid_fraction=min_valid_fraction,
    )
    if drop_last is None:
        drop_last = split == "train"
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        collate_fn=collate_loa,
        pin_memory=(device == "cuda"),
        persistent_workers=num_workers > 0,
        drop_last=drop_last,
    )


In [ ]:
DATASET = (
    "/global/cfs/cdirs/desi/users/zixinq/data/dr2_catalog/"
    "loa_spectra_preprocessed"
)

train_loader = make_loader(
    DATASET,
    split="train",
    batch_size=256,
    num_workers=0,       
    shuffle=True,
)


val_loader = make_loader(
    DATASET,
    split="val",
    batch_size=256,
    num_workers=0,
    shuffle=False,
    drop_last=False,
)

test_loader = make_loader(
    DATASET,
    split="test",
    batch_size=256,
    num_workers=0,
    shuffle=False,
    drop_last=False,
)

In [ ]:
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


# Keep this mapping fixed across training, validation, and inference.
CLASS_TO_IDX = {
    "SN Ia": 0,
    "SN II": 1,
    "SN Ib/c": 2,
    "TDE": 3,
    "afterglow": 4,
}


class TransientDataset(Dataset):
    def __init__(
        self,
        h5_path,
        use_weights=True,
        return_labels=False,
    ):
        self.return_labels = return_labels

        # Load into memory and close the file before DataLoader workers start.
        with h5py.File(h5_path, "r") as f:
            if not f.attrs.get("complete", False):
                raise ValueError("Preprocessing output is incomplete")

            required = {"X", "W", "M", "status", "targetid", "TYPE", "wave_grid"}
            missing = required - set(f.keys())
            if missing:
                raise ValueError(f"Missing datasets: {sorted(missing)}")

            status = f["status"][:]
            keep = status == 0

            self.X = f["X"][:][keep].astype(np.float32)
            self.M = f["M"][:][keep].astype(bool)
            stored_W = f["W"][:][keep].astype(np.float32)
            self.TARGETID = f["targetid"][:][keep].astype(np.int64)
            self.TYPE = f["TYPE"].asstr()[:][keep]
            self.wave = f["wave_grid"][:]
            self.source_indices = np.flatnonzero(keep)

        if not len(self.X):
            raise ValueError("No usable spectra (status == 0)")

        if (
            self.X.ndim != 2
            or self.X.shape != stored_W.shape
            or self.X.shape != self.M.shape
            or self.X.shape[1] != len(self.wave)
        ):
            raise ValueError("Flux, weights, masks, or wavelength shapes disagree")

        if (
            not np.all(np.isfinite(self.X[self.M]))
            or not np.all(np.isfinite(stored_W[self.M]))
            or np.any(stored_W[self.M] <= 0)
        ):
            raise ValueError("Invalid flux or weights at pixels marked valid")

        # Preserve preprocessing normalization; zero masked pixels.
        self.X = np.where(self.M, self.X, 0.0).astype(np.float32)

        # With weights disabled, valid pixels get 1; masked pixels stay 0.
        self.W = (
            np.where(self.M, stored_W, 0.0).astype(np.float32)
            if use_weights
            else self.M.astype(np.float32)
        )

        unknown = set(self.TYPE) - set(CLASS_TO_IDX)
        if unknown:
            raise ValueError(f"Unknown labels: {sorted(unknown)}")

        self.y = np.array(
            [CLASS_TO_IDX[label] for label in self.TYPE],
            dtype=np.int64,
        )

        print(
            f"Loaded {len(self.X)} usable spectra; "
            f"excluded {np.count_nonzero(~keep)} with nonzero status"
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        if self.return_labels:
            return self.X[i], self.W[i], self.y[i], self.TARGETID[i]
        return self.X[i], self.W[i], self.TARGETID[i]


trans_h5 = (
     "/global/cfs/cdirs/desi/users/zixinq/data/transient_catalog/transient_spectra_preprocessed.h5"
)



def collate_transient(batch):
    X, W, y, TID = zip(*batch)

    return {
        "x": torch.from_numpy(np.stack(X)),
        "w": torch.from_numpy(np.stack(W)),
        "y": torch.tensor(y, dtype=torch.long),
        "targetid": torch.tensor(np.asarray(TID, dtype=np.int64)),
        "spectype": [IDX_TO_CLASS[int(label)] for label in y],
    }


IDX_TO_CLASS = {idx: label for label, idx in CLASS_TO_IDX.items()}

trans_dataset = TransientDataset(
    trans_h5,
    use_weights=True,
    return_labels=True,
)

trans_loader = DataLoader(
    trans_dataset,
    batch_size=32,
    num_workers=0,
    pin_memory=(torch.device(device).type == "cuda"),
    shuffle=False,
    collate_fn=collate_transient,
)

batch = next(iter(trans_loader))
print(batch.keys())
print(batch["x"].shape)
print(batch["spectype"][:5])

In [ ]:
import os
import glob
import numpy as np
import torch
from torch.utils.data import IterableDataset, DataLoader


class SimulationNPZIterable(IterableDataset):
    def __init__(
        self,
        npz_files,
        use_weights=True,
    ):
        self.npz_files = list(npz_files)
        self.use_weights = use_weights
        
    def __iter__(self):
        for f in self.npz_files:

            with np.load(f, allow_pickle=True) as d:

                X = d["X"]

                W = (
                    d["W"]
                    if self.use_weights and "W" in d.files
                    else None
                )

                # TARGETID fallback
                if "TARGETID" in d.files:
                    targetid = d["TARGETID"]

                # Remove singleton channel dimension
                if X.ndim == 3:
                    X = X[:, 0, :]

                if W is not None and W.ndim == 3:
                    W = W[:, 0, :]

                keep = np.ones(
                    X.shape[0],
                    dtype=bool
                )

                for i in range(X.shape[0]):

                    if not keep[i]:
                        continue

                    x = torch.from_numpy(
                        np.asarray(X[i], dtype=np.float32)
                    )

                    if W is None:
                        w = None
                    else:
                        w = torch.from_numpy(
                            np.asarray(W[i], dtype=np.float32)
                        )

                    tid = targetid[i]

                    yield x, w, tid

sim_dir = (
    "/global/cfs/cdirs/desi/users/zixinq/data/transim/v6_preprocessed"
)

sim_files = sorted(
    glob.glob(
        os.path.join(sim_dir, "batch_*.npz")
    )
)

def collate_XW(batch):
    X, W, TID = zip(*batch)
    X = torch.from_numpy(np.stack(X, axis=0))              # (B, L)
    W = torch.from_numpy(np.stack(W, axis=0))              # (B, L)
    TID = np.asarray(TID, dtype=np.int64)                  # (B,)
    return X, W, TID


print("Number of simulation files:", len(sim_files))
print(sim_files[:3])

sim_dataset = SimulationNPZIterable(
    sim_files,
    use_weights=True,
)

sim_loader = DataLoader(
    sim_dataset,
    batch_size=32,
    num_workers=0,
    pin_memory=(device == "cuda"),
    collate_fn=collate_XW,
)

In [ ]:
def collect_mu(model, loader, device, max_batches=None):
    mus = []
    fluxes = []
    targetids = []
    spectypes = []

    model.eval()

    with torch.no_grad():
        for b, batch in enumerate(loader):
            if max_batches is not None and b >= max_batches:
                break
        
            #X = batch[0]
            #tid = batch[-1]

            X = batch["x"]
            tid = batch["targetid"]

            # Save the exact spectra used by the model
            fluxes.append(X.detach().cpu())
            spectypes.extend(batch["spectype"])
            '''
            x = X.to(
                device,
                non_blocking=True,
            ).unsqueeze(1)  # (B, 1, L)
            '''

            x = X.to(device, non_blocking=True)

            out = model(x)

            if isinstance(out, (tuple, list)):
                mu = out[1]
            else:
                mu = out["mu"]

            mus.append(mu.detach().cpu())

            if torch.is_tensor(tid):
                targetids.append(tid.detach().cpu())
            else:
                targetids.append(
                    torch.as_tensor(tid)
                )

    mu_all = torch.cat(mus, dim=0).numpy()
    flux_all = torch.cat(fluxes, dim=0).numpy()
    tid_all = torch.cat(targetids, dim=0).numpy()

    spectype_all = np.char.upper(np.char.strip(np.asarray(spectypes).astype(str)))
    
    return mu_all, flux_all, tid_all, spectype_all

In [ ]:
from pathlib import Path
import numpy as np
import torch

from pathlib import Path
import tempfile
import numpy as np
import torch


def collect_mu(model, loader, device, max_batches=None,
               output_dir="collected"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    names = ("mu", "flux", "targetid", "spectype")
    n_rows = 0
    n_chunks = 0
    schemas = {}
    model.eval()

    # Temporary batches are removed after the final files are written.
    with tempfile.TemporaryDirectory(dir=output_dir) as temp_dir:
        temp_dir = Path(temp_dir)

        with torch.no_grad():
            for batch in loader:
                X = batch["x"]

                # For an infinite dataset, set max_batches to bound the run.
                if max_batches is not None and n_chunks >= max_batches:
                    break

                out = model(X.to(device, non_blocking=True))
                mu = out[1] if isinstance(out, (tuple, list)) else out["mu"]

                tid = batch["targetid"]
                if torch.is_tensor(tid):
                    tid = tid.detach().cpu().numpy()
                else:
                    tid = np.asarray(tid)

                spectype = np.char.upper(
                    np.char.strip(np.asarray(batch["spectype"]).astype(str))
                )
                if any(len(s) > 64 for s in spectype):
                    raise ValueError("A spectype label exceeds 64 characters.")

                values = {
                    "mu": mu.detach().cpu().numpy(),
                    "flux": X.detach().cpu().numpy(),
                    "targetid": tid,
                    "spectype": spectype.astype("U64"),
                }

                for name, value in values.items():
                    if value.ndim == 0 or len(value) != len(X):
                        raise ValueError(f"Inconsistent batch length: {name}")

                    schema = (value.shape[1:], value.dtype)
                    if name in schemas and schemas[name] != schema:
                        raise ValueError(f"Shape or dtype changed: {name}")
                    schemas[name] = schema

                    np.save(
                        temp_dir / f"{n_chunks}_{name}.npy",
                        value,
                        allow_pickle=False,
                    )

                n_rows += len(X)
                n_chunks += 1

        if n_rows == 0:
            raise ValueError("No spectra were processed.")

        arrays = {}
        for name in names:
            trailing_shape, dtype = schemas[name]
            arrays[name] = np.lib.format.open_memmap(
                output_dir / f"{name}.npy",
                mode="w+",
                dtype=dtype,
                shape=(n_rows, *trailing_shape),
            )

            offset = 0
            for chunk in range(n_chunks):
                path = temp_dir / f"{chunk}_{name}.npy"
                block = np.load(path, mmap_mode="r")
                end = offset + len(block)
                arrays[name][offset:end] = block
                offset = end
                del block
                path.unlink()

            arrays[name].flush()

    np.save(output_dir / "n_rows.npy", np.asarray(n_rows))
    return tuple(arrays[name] for name in names)

In [ ]:
#train dataset latent space
mu_all, flux_all, tid_all, spectype_all = collect_mu(
    model,
    train_loader,
    device,
    output_dir="train_embeddings",
)

In [ ]:
mu_trans, flux_trans, tid_trans, spectype_trans = collect_mu(
    model,
    trans_loader,
    device,
)

In [ ]:
import os
import numpy as np

save_dir = "/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results"
os.makedirs(save_dir, exist_ok=True)


def to_numpy(x):
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.asarray(x)


np.savez_compressed(
    os.path.join(save_dir, "transient_latents.npz"),
    mu=np.asarray(mu_trans, dtype=np.float32),
    flux=np.asarray(flux_trans, dtype=np.float32),
    TARGETID=np.asarray(tid_trans),
)

In [ ]:
import os
import zipfile
import numpy as np

save_dir = "/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "dr2_train_latents.npz")


def write_selected(zf, key, array, indices, chunk_rows=256):
    """Write selected rows as an .npy member without a full RAM copy."""
    dtype = np.dtype(array.dtype)
    if dtype.hasobject:
        raise TypeError("Object arrays are not supported.")

    shape = (len(indices), *array.shape[1:])

    with zf.open(f"{key}.npy", mode="w", force_zip64=True) as destination:
        np.lib.format.write_array_header_2_0(
            destination,
            {
                "descr": np.lib.format.dtype_to_descr(dtype),
                "fortran_order": False,
                "shape": shape,
            },
        )

        for start in range(0, len(indices), chunk_rows):
            rows = indices[start:start + chunk_rows]
            block = np.ascontiguousarray(array[rows])
            destination.write(memoryview(block).cast("B"))


with zipfile.ZipFile(
    save_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=4,
    allowZip64=True,
) as zf:
    for label, suffix in [
        ("GALAXY", "galaxy"),
        ("QSO", "qso"),
        ("STAR", "star"),
    ]:
        indices = np.flatnonzero(spectype_all == label)

        for prefix, array in [
            ("mu", mu_all),
            ("tid", tid_all),
            ("flux", flux_all),
        ]:
            write_selected(zf, f"{prefix}_{suffix}", array, indices)

        print(label, (len(indices), *mu_all.shape[1:]))

print("Saved:", save_path)

In [ ]:
gal_mask = spectype_all == "GALAXY"
qso_mask = spectype_all == "QSO"
star_mask = spectype_all == "STAR"

mu_gal = mu_all[gal_mask]
mu_qso = mu_all[qso_mask]
mu_star = mu_all[star_mask]


flux_gal, tid_gal = flux_all[gal_mask], tid_all[gal_mask]
flux_qso, tid_qso = flux_all[qso_mask], tid_all[qso_mask]
flux_star, tid_star = flux_all[star_mask], tid_all[star_mask]

print("GALAXY:", mu_gal.shape)
print("QSO:", mu_qso.shape)
print("STAR:", mu_star.shape)

save_dir = "/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results"
np.savez_compressed(
    os.path.join(save_dir, "dr2_train_latents.npz"),
    mu_galaxy=mu_gal,
    tid_galaxy=tid_gal,
    flux_galaxy = flux_gal,

    mu_qso=mu_qso,
    tid_qso=tid_qso,
    flux_qso = flux_qso,

    mu_star=mu_star,
    tid_star=tid_star,
    flux_star= flux_star,
)

In [ ]:
#test dataset latent space
mu_all, flux_all, tid_all, spectype_all = collect_mu(
    model, test_loader, device
)

gal_mask = spectype_all == "GALAXY"
qso_mask = spectype_all == "QSO"
star_mask = spectype_all == "STAR"

mu_gal = mu_all[gal_mask]
mu_qso = mu_all[qso_mask]
mu_star = mu_all[star_mask]


flux_gal, tid_gal = flux_all[gal_mask], tid_all[gal_mask]
flux_qso, tid_qso = flux_all[qso_mask], tid_all[qso_mask]
flux_star, tid_star = flux_all[star_mask], tid_all[star_mask]

print("GALAXY:", mu_gal.shape)
print("QSO:", mu_qso.shape)
print("STAR:", mu_star.shape)

In [ ]:
STAR_DATASET = "/global/cfs/cdirs/desi/users/zixinq/data/dr2_catalog/loa_spectra_preprocesseds" 

star_loader = make_loader(
    STAR_DATASET,
    split="test", 
    batch_size=256,
    num_workers=4,
    shuffle=False,
    drop_last=False
)

In [ ]:
mu_star, flux_star, tid_star, spectype_star = collect_mu(model, star_loader, device)

print("STAR shape:", mu_star.shape)
print("STAR Target IDs:", tid_star.shape)

In [ ]:
save_dir = "/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results"
np.savez_compressed(
    os.path.join(save_dir, "dr2_test_latents.npz"),
    mu_galaxy=mu_gal,
    tid_galaxy=tid_gal,
    flux_galaxy = flux_gal,

    mu_qso=mu_qso,
    tid_qso=tid_qso,
    flux_qso = flux_qso,

    mu_star=mu_star,
    tid_star=tid_star,
    flux_star= flux_star,
)

In [ ]:
import numpy as np
import os

save_dir = "/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results"

with np.load(
    os.path.join(save_dir, "transient_latents.npz"),
    allow_pickle=False,
) as data:
    mu_tr = data["mu"]
    flux_tr = data["flux"]
    tid_tr = data["TARGETID"]

with np.load(
    os.path.join(save_dir, "simulation_latents.npz"),
    allow_pickle=False,
) as data:
    mu_sim = data["mu"]
    flux_sim = data["flux"]
    tid_sim = data["TARGETID"]

In [ ]:
with np.load(
    os.path.join(save_dir, "dr2_test_latents.npz"),
    allow_pickle=False,
) as data:
    mu_galaxy_test = data["mu_galaxy"]
    flux_galaxy_test = data["flux_galaxy"]
    tid_galaxy_test = data["tid_galaxy"]

    mu_qso_test = data["mu_qso"]
    tid_qso_test = data["tid_qso"]
    flux_qso_test = data["flux_qso"]

    mu_star_test = data["mu_star"]
    tid_star_test = data["tid_star"]
    flux_star_test = data["flux_star"]

In [ ]:
with np.load(
    os.path.join(save_dir, "dr2_train_latents.npz"),
    allow_pickle=False,
) as data:
    mu_galaxy = data["mu_galaxy"]
    flux_galaxy = data["flux_galaxy"]
    tid_galaxy = data["tid_galaxy"]

    mu_qso = data["mu_qso"]
    tid_qso = data["tid_qso"]
    flux_qso = data["flux_qso"]

    mu_star = data["mu_star"]
    tid_star = data["tid_star"]
    flux_star = data["flux_star"]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
import umap  # Install if needed: %pip install umap-learn

SEED = 42
MAX_REFERENCE_PER_CLASS = 5000
MAX_PLOT_PER_CLASS = 2000

rng = np.random.default_rng(SEED)

# Original VAE training populations
training = {
    "Galaxy": (mu_galaxy, tid_galaxy),
    "QSO": (mu_qso, tid_qso),
    "Star": (mu_star, tid_star),
}

# Populations to display after transforming.
display_data = {
    "Galaxy (test)": (mu_galaxy_test, tid_galaxy_test),
    "QSO (test)": (mu_qso_test, tid_qso_test),
    "Star (test)": (mu_star_test, tid_star_test),
    "Real transient": (mu_tr, tid_tr),
}

# Check the full supplied IDs before subsampling.
train_ids = np.concatenate([
    np.asarray(ids).reshape(-1) for _, ids in training.values()
])

for name, (_, ids) in display_data.items():
    overlap = np.intersect1d(train_ids, np.asarray(ids).reshape(-1))
    print(f"{name}: {len(overlap):,} unique TARGETIDs also in training")
    if name.endswith("(test)") and len(overlap):
        raise ValueError(f"{name} overlaps training; check your original split.")

# Validate dimensions and retain finite rows with their matching IDs.
latent_dim = np.asarray(mu_galaxy).shape[1]

def clean_latents(mu, ids):
    mu = np.asarray(mu)
    ids = np.asarray(ids).reshape(-1)
    if mu.ndim != 2 or mu.shape[1] != latent_dim or len(mu) != len(ids):
        raise ValueError("Expected matching (N, latent_dim) means and N IDs.")
    valid = np.isfinite(mu).all(axis=1)
    return mu[valid], ids[valid]

training = {name: clean_latents(*values)
            for name, values in training.items()}
display_data = {name: clean_latents(*values)
                for name, values in display_data.items()}

if any(len(mu) == 0 for mu, _ in [*training.values(), *display_data.values()]):
    raise ValueError("A population has no finite latent vectors.")

n_reference = min(
    MAX_REFERENCE_PER_CLASS,
    *(len(mu) for mu, _ in training.values()),
)
X_reference = np.concatenate([
    mu[rng.choice(len(mu), n_reference, replace=False)]
    for mu, _ in training.values()
])
rng.shuffle(X_reference)

if len(X_reference) < 4:
    raise ValueError("Too few training vectors for a useful projection.")

plot_arrays, plot_labels, plot_ids = [], [], []
names = list(display_data)

for j, (name, (mu, ids)) in enumerate(display_data.items()):
    take = rng.choice(
        len(mu), min(MAX_PLOT_PER_CLASS, len(mu)), replace=False
    )
    plot_arrays.append(mu[take])
    plot_labels.append(np.full(len(take), j))
    plot_ids.append(ids[take])

X_plot = np.concatenate(plot_arrays)
y_plot = np.concatenate(plot_labels)
ids_plot = np.concatenate(plot_ids)

# Fit only on training reference vectors. Use raw latent coordinates.
pca = PCA(n_components=2, svd_solver="full")
pca.fit(X_reference)
pca_xy = pca.transform(X_plot)
explained = pca.explained_variance_ratio_

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=min(30, len(X_reference) - 1),
    min_dist=0.1,
    metric="euclidean",
    random_state=SEED,
    transform_seed=SEED,
    n_jobs=1,
)
reducer.fit(X_reference)  # No class labels supplied.
umap_xy = reducer.transform(X_plot)

print(f"Reference: {n_reference:,} training rows per class")
print(f"PCA retained variance: {explained.sum():.1%}")



colors = ["#226E9C", "#55A868", "#AF58BA", "#FDBD3C", "#C44E52"]
markers = ["o", "s", "D", "^", "v"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.8))

for ax, coordinates in zip(axes, [pca_xy, umap_xy]):
    for j, (color, marker) in enumerate(zip(colors, markers)):
        mask = y_plot == j
        transient_like = j >= 3
        ax.scatter(
            coordinates[mask, 0], coordinates[mask, 1],
            s=24 if transient_like else 9,
            marker=marker,
            facecolors="none" if transient_like else color,
            edgecolors=color if transient_like else "none",
            linewidths=0.7 if transient_like else 0,
            alpha=0.75 if transient_like else 0.3,
            rasterized=True,
        )
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set(
    title="(a) PCA",
    xlabel=f"PC1 ({explained[0]:.1%} reference variance)",
    ylabel=f"PC2 ({explained[1]:.1%} reference variance)",
)
axes[1].set(title="(b) UMAP", xlabel="UMAP 1", ylabel="UMAP 2")

handles = [
    Line2D(
        [], [], linestyle="none", marker=markers[j], color=colors[j],
        markerfacecolor="none" if j >= 3 else colors[j],
        label=f"{name} (n={(y_plot == j).sum():,})",
    )
    for j, name in enumerate(names)
]
fig.legend(
    handles=handles, loc="upper center",
    bbox_to_anchor=(0.5, 0.93), ncol=3, frameon=False,
)
fig.suptitle("VAE latent means · projections fitted on DR2 training spectra")
fig.tight_layout(rect=(0, 0, 1, 0.81))
plt.show()

In [ ]:
rng = np.random.default_rng(42)

import numpy as np

def sample_array(mu, tid, flux, n, seed=42):
    assert len(mu) == len(tid) == len(flux), (
        "mu, tid, and flux must have matching rows"
    )

    rng = np.random.default_rng(seed)
    idx = rng.choice(len(mu), size=min(n, len(mu)), replace=False)

    return mu[idx], tid[idx], flux[idx], idx
    
mu_gal_s, tid_gal_s, flux_gal_s, idx_gal = sample_array(
    mu_galaxy, tid_galaxy, flux_galaxy, 5_000_000
)

mu_qso_s, tid_qso_s, flux_qso_s, idx_qso = sample_array(
    mu_qso, tid_qso, flux_qso, 500_000
)

mu_tr_s, tid_tr_s, flux_tr_s, idx_tr = sample_array(
    mu_tr, tid_tr, flux_tr, 5000
)

mu_sim_s, tid_sim_s, flux_sim_s, idx_sim = sample_array(
    mu_sim, tid_sim, flux_sim, 50_000
)

In [ ]:
X_real = np.vstack([
    mu_gal_s,
    mu_qso_s,
    mu_tr_s,
])

labels_real = np.concatenate([
    np.full(len(mu_gal_s), "Galaxy"),
    np.full(len(mu_qso_s), "QSO"),
    np.full(len(mu_tr_s), "Real transient"),
])


In [ ]:
import numpy as np
import pandas as pd
import umap
import matplotlib.pyplot as plt
import seaborn as sns


X_real = np.vstack([
    mu_gal_s,
    mu_qso_s,
    mu_tr_s,
])

labels_real = np.concatenate([
    np.full(len(mu_gal_s), "Galaxy"),
    np.full(len(mu_qso_s), "QSO"),
    np.full(len(mu_tr_s), "Real transient"),
])

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=50,
    min_dist=0.2,
    metric="euclidean",
    random_state=42,
)

Z_real = reducer.fit_transform(X_real)
Z_sim = reducer.transform(mu_sim_s)

In [ ]:

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns

plt.figure(figsize=(10, 8))
ax = plt.gca()


# galaxy
mask = labels_real == "Galaxy"
plt.scatter(
    Z_real[mask, 0],
    Z_real[mask, 1],
    s=3,
    alpha=0.3,
    label="Galaxy",
    zorder=1,
)

# QSO
mask = labels_real == "QSO"
plt.scatter(
    Z_real[mask, 0],
    Z_real[mask, 1],
    s=3,
    alpha=0.5,
    label="QSO",
    zorder=2,
)
'''
# simulations
plt.scatter(
    Z_sim[:, 0],
    Z_sim[:, 1],
    s=5,
    alpha=0.2,
    label="Simulated transient",
    zorder=1,
)
'''

# real transients 
mask = labels_real == "Real transient"
plt.scatter(
    Z_real[mask, 0],
    Z_real[mask, 1],
    s=6,
    alpha=0.8,
    label="Real transient",
    zorder=2,
)


plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.savefig('vae_umap16.png', dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def as_2d_flux(flux):
    flux = np.asarray(flux)
    if flux.ndim == 3 and flux.shape[1] == 1:
        flux = flux[:, 0, :]
    if flux.ndim != 2:
        raise ValueError(f"Expected (N, L) or (N, 1, L), got {flux.shape}")
    return flux

groups = {
    "Galaxy": (
        Z_real[labels_real == "Galaxy"],
        as_2d_flux(flux_gal_s),
    ),
    "QSO": (
        Z_real[labels_real == "QSO"],
        as_2d_flux(flux_qso_s),
    ),
    "Real transient": (
        Z_real[labels_real == "Real transient"],
        as_2d_flux(flux_tr_s),
    ),
    "Simulated transient": (
        Z_sim,
        as_2d_flux(flux_sim_s),
    ),
}

for name, (coords, flux) in groups.items():
    assert len(coords) == len(flux), (
        f"{name}: {len(coords)} UMAP points but {len(flux)} spectra. "
        "Apply the same sampling indices to both."
    )

In [ ]:
def inspect_umap_region(
    groups, xlim, ylim, wave=None,
    n_per_class=5, seed=42, panel_size=(3.5, 2.3),
):
    rng = np.random.default_rng(seed)

    fig, axes = plt.subplots(
        len(groups), n_per_class,
        figsize=(
            panel_size[0] * n_per_class,
            panel_size[1] * len(groups),
        ),
        squeeze=False,
        sharex=True,
    )

    selected = {}

    for row, (name, (coords, flux)) in enumerate(groups.items()):
        inside = (
            (coords[:, 0] >= xlim[0])
            & (coords[:, 0] <= xlim[1])
            & (coords[:, 1] >= ylim[0])
            & (coords[:, 1] <= ylim[1])
        )

        candidates = np.flatnonzero(inside)
        chosen = rng.choice(
            candidates,
            size=min(n_per_class, len(candidates)),
            replace=False,
        )
        selected[name] = chosen

        print(f"{name}: {len(candidates):,} spectra in region")

        x = np.arange(flux.shape[1]) if wave is None else np.asarray(wave)
        if x.shape != (flux.shape[1],):
            raise ValueError(f"Wavelength grid does not match {name} spectra")

        for col, ax in enumerate(axes[row]):
            if col >= len(chosen):
                ax.axis("off")
                continue

            idx = chosen[col]
            ax.plot(x, flux[idx], lw=0.8)
            ax.set_title(
                f"{name} | index {idx}\n"
                f"UMAP ({coords[idx, 0]:.2f}, {coords[idx, 1]:.2f})",
                fontsize=9,
            )
            ax.set_ylabel("Flux")
            ax.set_xlabel("Pixel" if wave is None else "Wavelength (Å)")

    fig.tight_layout()
    plt.show()
    return selected

In [ ]:
xlim = (7, 9)
ylim = (3, 5)

selected = inspect_umap_region(
    groups,
    xlim=xlim,
    ylim=ylim,
    wave=None,       
    n_per_class=6,
)

In [ ]:
xlim = (8, 10)
ylim = (13, 17)

selected = inspect_umap_region(
    groups,
    xlim=xlim,
    ylim=ylim,
    wave=None,       
    n_per_class=10,
)

In [ ]:
#cornerplot

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import corner

#DIMS = [0, 1, 2, 3, 4, 5]
DIMS = list(range(16))  # Full plot; much larger.

if "mu_qso" not in globals():
    raise RuntimeError(
        "Create mu_qso first by encoding QSO spectra with the same VAE "
        "and preprocessing used for galaxies and transients."
    )

groups = [
    ("Galaxies", mu_galaxy, "#356A9A", "-"),
    ("QSOs", mu_qso, "#7953A2", ":"),
    ("Transients", mu_tr, "#C45E16", "--"),
]

prepared = []
for name, mu, color, style in groups:
    values = np.asarray(mu)[:, DIMS]
    valid = np.isfinite(values).all(axis=1)
    print(f"{name}: {valid.sum():,} valid; {(~valid).sum():,} removed")
    values = values[valid]

    if len(values) <= len(DIMS):
        raise ValueError(f"Too few valid spectra for {name}")

    prepared.append((name, values, color, style))

# Shared ranges and bins; retain the tails.
combined = np.concatenate([values for _, values, _, _ in prepared])
lo, hi = combined.min(axis=0), combined.max(axis=0)
pad = np.where(hi > lo, 0.03 * (hi - lo), 0.5)
limits = list(zip(lo - pad, hi + pad))

fig = None
handles = []

for name, values, color, style in prepared:
    fig = corner.corner(
        values,
        fig=fig,
        labels=[rf"$\mu_{{{d}}}$" for d in DIMS],
        range=limits,
        bins=30,
        color=color,
        max_n_ticks=3,
        plot_datapoints=False,
        plot_density=False,
        plot_contours=True,
        no_fill_contours=True,
        levels=[0.68, 0.95],
        smooth=1.0,
        hist_kwargs={
            "density": True,
            "linewidth": 1.6,
            "linestyle": style,
        },
        contour_kwargs={
            "linewidths": 1.3,
            "linestyles": style,
        },
    )

    handles.append(
        Line2D([], [], color=color, linestyle=style, linewidth=1.6,
               label=f"{name} (n={len(values):,})")
    )

fig.legend(
    handles=handles,
    loc="upper right",
    bbox_to_anchor=(0.97, 0.96),
    frameon=False,
)
fig.suptitle("VAE latent means: galaxies, QSOs and transients", y=1.02)

fig.savefig("galaxy_qso_transient_corner.pdf", bbox_inches="tight")
fig.savefig("galaxy_qso_transient_corner.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path


SAVE_DIR = Path(globals().get("save_dir", "/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results/"))
OUTPUT_DIR = Path(globals().get("save_dir", "/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results/"))
SEED = 42

MAX_ROWS_PER_CLASS = 10000 
SCOPE = "Exploratory sample; split not restricted"

# Optional: boolean arrays aligned to ORIGINAL NPZ rows, not sampled rows.
# ROW_MASKS = {"galaxy": galaxy_test_mask, "qso": qso_test_mask,
#              "transient": transient_test_mask, "simulation": simulation_test_mask}
ROW_MASKS = {}

PAIRS = [(0, 1), (0, 2), (1, 2), (3, 4), (4, 5), (2, 5)]

In [ ]:
"""Plot saved DESI VAE means; imported by VAE_latent_diagnostics.ipynb.

No model training, anomaly selection, or automatic test-split inference.
"""
from pathlib import Path
import csv
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

SOURCES = {
    'galaxy': ('dr2_latents.npz', 'mu_galaxy', 'tid_galaxy'),
    'qso': ('dr2_latents.npz', 'mu_qso', 'tid_qso'),
    'transient': ('transient_latents.npz', 'mu', 'TARGETID'),
    'simulation': ('simulation_latents.npz', 'mu', 'TARGETID'),
}
NAMES = ['Galaxies', 'QSOs', 'Real transients', 'Simulations']
COLORS = ['#356A9A', '#7953A2', '#C45E16', '#27816B']
STYLES = ['-', ':', '--', '-.']
MARKERS = ['o', 's', '^', 'D']


def load_groups(save_dir, max_rows=10000, seed=42, row_masks=None):
    """Sample finite rows uniformly per supplied class, retaining source row IDs.

    row_masks maps class keys to boolean arrays in ORIGINAL NPZ row order.
    Omitted classes use all rows. NPZ members decompress in full, one group at a
    time; the cap limits retained arrays and plotting, not decompression memory.
    """
    if max_rows is not None and max_rows < 2:
        raise ValueError('max_rows must be at least 2 or None')
    row_masks = {} if row_masks is None else row_masks
    if set(row_masks) - set(SOURCES):
        raise ValueError('Unknown class key in row_masks')
    rng = np.random.default_rng(seed)
    groups, ncode = {}, None
    for (key, (file, mu_key, id_key)), name, color, style, marker in zip(
            SOURCES.items(), NAMES, COLORS, STYLES, MARKERS):
        with np.load(Path(save_dir) / file, allow_pickle=False) as data:
            mu = data[mu_key]
            ids = data[id_key].reshape(-1)
        if mu.ndim != 2 or len(ids) != len(mu):
            raise ValueError(f'{key}: expected mu=(N,D) and matching IDs')
        if ncode is None:
            ncode = mu.shape[1]
        if mu.shape[1] != ncode or ncode < 2:
            raise ValueError('Groups must have the same latent dimension >= 2')
        mask = np.asarray(row_masks.get(key, np.ones(len(mu), dtype=bool)))
        if mask.dtype != bool or mask.shape != (len(mu),):
            raise ValueError(f'{key}: mask must be boolean with shape ({len(mu)},)')
        finite = np.isfinite(mu).all(axis=1)
        eligible = np.flatnonzero(mask & finite)
        if len(eligible) < 2:
            raise ValueError(f'{key}: fewer than two eligible finite rows')
        rows = eligible if max_rows is None else np.sort(rng.choice(
            eligible, min(max_rows, len(eligible)), replace=False))
        groups[key] = dict(mu=mu[rows].astype(float), ids=ids[rows], rows=rows,
                           name=name, color=color, style=style, marker=marker,
                           file=file, mu_key=mu_key, total=len(mu),
                           eligible=len(eligible), invalid=int((mask & ~finite).sum()))
        print(f'{name}: {len(rows):,} plotted / {len(eligible):,} eligible; '
              f'{groups[key]["invalid"]:,} non-finite rows removed')
    return groups


def save_selection(groups, output_dir, seed, scope):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    with (out / 'sampled_rows.csv').open('w', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['group', 'sample_index', 'source_file', 'source_row', 'TARGETID'])
        for key, g in groups.items():
            writer.writerows((key, i, g['file'], int(row), str(tid))
                             for i, (row, tid) in enumerate(zip(g['rows'], g['ids'])))
    (out / 'selection.json').write_text(json.dumps(dict(
        seed=seed, scope=scope,
        groups={k: {field: g[field] for field in
                    ['file', 'mu_key', 'total', 'eligible', 'invalid']} for k, g in groups.items()}
    ), indent=2))


def grid(n):
    fig, axes = plt.subplots(int(np.ceil(n / 4)), 4,
                             figsize=(15, 3 * int(np.ceil(n / 4))), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.ravel()[:n]


def legend(fig, groups):
    fig.legend(handles=[Line2D([], [], color=g['color'], linestyle=g['style'],
                              label=f'{g["name"]} (n={len(g["mu"]):,})')
                        for g in groups.values()],
               loc='upper center', bbox_to_anchor=(0.5, 0.965), ncol=4, frameon=False)


def histograms(groups, bins=50):
    d = next(iter(groups.values()))['mu'].shape[1]
    fig, axes = grid(d)
    for j, ax in enumerate(axes):
        values = np.concatenate([g['mu'][:, j] for g in groups.values()])
        edges = np.histogram_bin_edges(values, bins=bins)
        for g in groups.values():
            ax.hist(g['mu'][:, j], bins=edges, density=True, histtype='step',
                    color=g['color'], linestyle=g['style'], linewidth=1.5)
        ax.set(xlabel=f'μ{j}', ylabel='Probability density')
        ax.grid(alpha=0.12)
    fig.suptitle('Each latent dimension · separate density normalization per class')
    legend(fig, groups)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    return fig


def boxplots(groups):
    d = next(iter(groups.values()))['mu'].shape[1]
    fig, axes = grid(d)
    for j, ax in enumerate(axes):
        artists = ax.boxplot([g['mu'][:, j] for g in groups.values()],
                            patch_artist=True, showfliers=False, whis=(5, 95))
        for patch, g in zip(artists['boxes'], groups.values()):
            patch.set_facecolor(g['color'])
            patch.set_alpha(0.55)
        ax.set_xticks(np.arange(1, 5), ['Gal', 'QSO', 'Real', 'Sim'])
        ax.set_ylabel(f'μ{j}')
        ax.grid(axis='y', alpha=0.12)
    fig.suptitle('Latent distributions · boxes: 25–75%; center: median; whiskers: 5–95%\n'
                 'Tails beyond whiskers are omitted')
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    return fig


def pairwise_scatter(groups, pairs=((0, 1), (0, 2), (1, 2), (3, 4), (4, 5), (2, 5)),
                     max_points=1000, seed=42):
    d = next(iter(groups.values()))['mu'].shape[1]
    if not pairs or any(a == b or min(a, b) < 0 or max(a, b) >= d for a, b in pairs):
        raise ValueError('Select valid, distinct dimension pairs')
    count = min(max_points, *(len(g['mu']) for g in groups.values()))
    if count < 1:
        raise ValueError('max_points must be positive')
    rng = np.random.default_rng(seed)
    samples = {k: g['mu'][rng.choice(len(g['mu']), count, replace=False)]
               for k, g in groups.items()}
    fig, axes = plt.subplots(int(np.ceil(len(pairs) / 3)), 3,
                             figsize=(15, 4.3 * int(np.ceil(len(pairs) / 3))), squeeze=False)
    for ax, (a, b) in zip(axes.flat, pairs):
        for k, g in groups.items():
            x = samples[k]
            ax.scatter(x[:, a], x[:, b], s=13, marker=g['marker'],
                       color=g['color'], alpha=0.3, linewidths=0, rasterized=True,
                       label=g['name'])
        ax.set(xlabel=f'μ{a}', ylabel=f'μ{b}')
        ax.grid(alpha=0.12)
    for ax in list(axes.flat)[len(pairs):]:
        ax.set_visible(False)
    axes.flat[0].legend(frameon=False, fontsize=9)
    fig.suptitle(f'Pairwise latent means · {count:,} random rows per class; seed={seed}')
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig


def correlations(groups):
    fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
    cmap = plt.get_cmap('RdBu_r').copy()
    cmap.set_bad('#CCCCCC')
    for ax, g in zip(axes.flat, groups.values()):
        x = g['mu']
        sd = x.std(axis=0)
        valid = sd > 0
        corr = np.full((x.shape[1], x.shape[1]), np.nan)
        if valid.sum():
            z = (x[:, valid] - x[:, valid].mean(axis=0)) / sd[valid]
            corr[np.ix_(valid, valid)] = np.clip(z.T @ z / len(z), -1, 1)
        im = ax.imshow(corr, vmin=-1, vmax=1, cmap=cmap)
        ax.set(title=f'{g["name"]} · n={len(x):,}', xlabel='Latent dimension',
               ylabel='Latent dimension', xticks=range(x.shape[1]), yticks=range(x.shape[1]))
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label='Pearson correlation')
    fig.suptitle('Within-class latent correlations · gray = undefined (constant dimension)')
    return fig


def latent_heatmap(groups, max_rows=80, seed=42):
    """One shared scale based on the sampled galaxies; descriptive, not a fitted classifier."""
    reference = groups['galaxy']['mu']
    center, scale = reference.mean(axis=0), reference.std(axis=0)
    scale = np.where(scale > 0, scale, np.nan)
    count = min(max_rows, *(len(g['mu']) for g in groups.values()))
    if count < 1:
        raise ValueError('max_rows must be positive')
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(1, 4, figsize=(15, 8), constrained_layout=True)
    selected = {}
    cmap = plt.get_cmap('RdBu_r').copy()
    cmap.set_bad('#CCCCCC')
    for ax, (key, g) in zip(axes, groups.items()):
        idx = rng.choice(len(g['mu']), count, replace=False)
        selected[key] = idx
        z = (g['mu'][idx] - center) / scale
        im = ax.imshow(z, aspect='auto', interpolation='nearest',
                       cmap=cmap, vmin=-3, vmax=3)
        ax.set(title=g['name'], xlabel='Latent dimension',
               xticks=range(z.shape[1]))
    axes[0].set_ylabel('Random sampled row (order is not shared across classes)')
    fig.colorbar(im, ax=axes.tolist(), extend='both', shrink=0.7,
                 label='(μ − galaxy mean) / galaxy SD; colors saturate at ±3')
    fig.suptitle(f'Latent values · {count} rows per class · gray = constant galaxy dimension')
    return fig, selected


def latent_traversals(model, groups, anchor_group='galaxy', anchor_index=0,
                      reference_group='galaxy', wavelength=None):
    """Vary one coordinate of a real saved latent vector at a time.

    Uses reference-group marginal 10/50/90 percentiles. Other coordinates stay
    fixed; combinations may leave the learned data distribution.
    """
    import torch
    anchor = groups[anchor_group]['mu'][anchor_index]
    reference = groups[reference_group]['mu']
    levels = np.quantile(reference, [0.1, 0.5, 0.9], axis=0)
    d = len(anchor)
    if hasattr(model, 'ncode') and model.ncode != d:
        raise ValueError('Model latent dimension does not match saved vectors')
    param = next(model.parameters())
    was_training = model.training
    model.eval()
    reconstructions = []
    try:
        with torch.inference_mode():
            for j in range(d):
                z = np.tile(anchor, (4, 1))
                z[1:, j] = levels[:, j]
                y = model.decode(torch.as_tensor(z, device=param.device, dtype=param.dtype))
                if y.ndim == 3 and y.shape[1] == 1:
                    y = y[:, 0, :]
                if y.ndim != 2 or y.shape[0] != 4 or not torch.isfinite(y).all():
                    raise ValueError('Decoder must return finite (B,L) or (B,1,L) spectra')
                reconstructions.append(y.detach().cpu().numpy())
    finally:
        model.train(was_training)
    reconstructions = np.stack(reconstructions)
    length = reconstructions.shape[-1]
    if wavelength is None:
        wave, label = np.arange(length), 'Input pixel index'
    else:
        wave = np.asarray(wavelength)
        if wave.shape != (length,) or not np.isfinite(wave).all() or not np.all(np.diff(wave) > 0):
            raise ValueError('Wavelength must be a finite increasing 1D decoder input grid')
        label = 'Wavelength (units of supplied grid)'
    fig, axes = grid(d)
    for j, ax in enumerate(axes):
        ax.plot(wave, reconstructions[j, 0], color='#333333', lw=1.3, label='Anchor')
        for k, (color, style, quantile) in enumerate(zip(
                ['#356A9A', '#C45E16', '#7953A2'], ['--', ':', '-.'], [10, 50, 90])):
            ax.plot(wave, reconstructions[j, k + 1], color=color, linestyle=style,
                    lw=1, label=f'Reference p{quantile}')
        ax.set(title=f'Vary μ{j} (anchor={anchor[j]:.2g})', xlabel=label,
               ylabel='Decoded normalized flux')
    axes[0].legend(fontsize=8, frameon=False)
    fig.suptitle(f'One-coordinate traversals · anchor: {anchor_group}, sample {anchor_index}\n'
                 f'10/50/90 percentiles from {reference_group}; all other coordinates fixed')
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    return fig, dict(decoded=reconstructions, anchor=anchor, quantile_values=levels,
                     wavelength=wave, anchor_source_row=groups[anchor_group]['rows'][anchor_index],
                     anchor_TARGETID=groups[anchor_group]['ids'][anchor_index],
                     anchor_group=anchor_group, reference_group=reference_group)


def preview_figure(fig, scope='Exploratory sample'):
    """Display inline without writing files; retain the figure for optional export."""
    from IPython.display import display
    fig.text(0.5, -0.01, scope, ha='center', fontsize=10)
    display(fig)
    plt.close(fig)  # Avoid duplicate automatic output; fig remains saveable.
    return fig


def export_figure(fig, output_dir, name):
    """Save a previously previewed figure without replotting or resampling."""
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f'{name}.png', dpi=160, bbox_inches='tight')
    fig.savefig(out / f'{name}.pdf', bbox_inches='tight')


In [ ]:
groups = load_groups(SAVE_DIR, MAX_ROWS_PER_CLASS, SEED, ROW_MASKS)
figures = {}
plot_data = {}

In [ ]:
figures["01_histograms"] = preview_figure(
    histograms(groups), SCOPE
)

In [ ]:
figures["02_boxplots"] = preview_figure(
    boxplots(groups), SCOPE
)

In [ ]:
figures["03_pairwise_scatter"] = preview_figure(
    pairwise_scatter(groups, PAIRS, max_points=1000, seed=SEED), SCOPE
)

In [ ]:
df = pd.read_csv("/global/cfs/cdirs/desicollab/users/xhall/data/share/dtc_validation_set_2026-09-27/manifest.csv")
df

In [ ]:
#reconstruction error distribution
import numpy as np
import torch


def evaluate_test_reconstruction(model, loader):
    """Evaluate all loader rows; retain only scalar metrics, not full spectra."""
    if torch.distributed.is_initialized() and torch.distributed.get_world_size() > 1:
        raise RuntimeError("Run in one process so this loader covers the full test split.")
    parameter = next(model.parameters())
    was_training = model.training
    records = {key: [] for key in (
        "TARGETID", "spectype", "n_valid", "sse", "mse", "rmse",
        "sqrt_weight_sum", "reconstruction_term", "sqrt_weighted_mse",
    )}
    model.eval()
    try:
        with torch.inference_mode():
            for batch in loader:
                x = batch["x"].to(device=parameter.device, dtype=parameter.dtype)
                mask = batch["mask"].to(device=parameter.device, dtype=torch.bool)
                weights = batch["weight"]
                if weights is None:
                    raise ValueError("Use include_weights=True for this evaluation.")
                w = weights.to(device=parameter.device, dtype=parameter.dtype)
                if x.ndim != 3 or x.shape[1] != 1 or mask.shape != x.shape or w.shape != x.shape:
                    raise ValueError("Expected matching (B, 1, L) input, mask, and weight arrays.")
                # Do not silently change the encoder inputs or discard failed predictions.
                if not torch.isfinite(x).all():
                    raise ValueError("Non-finite model inputs; check preprocessing before evaluation.")
                if not torch.isfinite(w[mask]).all():
                    raise ValueError("Non-finite weights at valid pixels.")
                mu, _ = model.encode(x)
                xhat = model.decode(mu)  # forward() still samples z, even in eval mode.
                if xhat.shape != x.shape or not torch.isfinite(xhat).all():
                    raise ValueError("Decoder output has the wrong shape or non-finite values.")
                residual2 = (x.double() - xhat.double()).square()
                dims = (1, 2)
                n_valid = mask.sum(dim=dims)
                sse = torch.where(mask, residual2, 0.0).sum(dim=dims)
                mse = sse / n_valid.clamp_min(1)
                mse = torch.where(n_valid > 0, mse, torch.nan)
                # Same pixel weighting and factor as VAE.loss; additionally respect M.
                a = torch.where(mask, w.double().clamp_min(0).sqrt(), 0.0)
                a_sum = a.sum(dim=dims)
                term = 0.5 * (a * residual2).sum(dim=dims)
                wmse = 2.0 * term / a_sum.clamp_min(torch.finfo(torch.float64).tiny)
                wmse = torch.where(a_sum > 0, wmse, torch.nan)
                records["TARGETID"].append(batch["targetid"].cpu().numpy())
                records["spectype"].append(np.asarray(batch["spectype"], dtype=str))
                for key, value in dict(n_valid=n_valid, sse=sse, mse=mse,
                                       rmse=mse.sqrt(), sqrt_weight_sum=a_sum,
                                       reconstruction_term=term,
                                       sqrt_weighted_mse=wmse).items():
                    records[key].append(value.cpu().numpy())
    finally:
        model.train(was_training)
    if not records["TARGETID"]:
        raise ValueError("The test loader returned no spectra.")
    result = {key: np.concatenate(value) for key, value in records.items()}
    valid = result["n_valid"] > 0
    n_pixels = int(result["n_valid"].sum())
    total_weight = float(result["sqrt_weight_sum"].sum())
    total_term = float(result["reconstruction_term"].sum())
    pixel_mse = float(result["sse"].sum() / n_pixels) if n_pixels else float("nan")
    summary = {
        "n_spectra": len(valid),
        "n_spectra_without_valid_pixels": int((~valid).sum()),
        "n_spectra_without_positive_weight": int((result["sqrt_weight_sum"] == 0).sum()),
        "n_valid_pixels": n_pixels,
        "pixel_averaged_mse": pixel_mse,
        "pixel_averaged_rmse": float(np.sqrt(pixel_mse)),
        "mean_per_spectrum_mse": float(result["mse"][valid].mean()) if valid.any() else float("nan"),
        "median_per_spectrum_mse": float(np.median(result["mse"][valid])) if valid.any() else float("nan"),
        "sqrt_weighted_mse": 2.0 * total_term / total_weight if total_weight > 0 else float("nan"),
        "reconstruction_term_sum": total_term,
        "reconstruction_term_mean_per_spectrum": total_term / len(valid),
    }
    return result, summary


In [ ]:
test_errors, test_error_summary = evaluate_test_reconstruction(model, test_loader)
wmse_test = test_errors["sqrt_weighted_mse"]
spectype_test = np.char.upper(np.char.strip(test_errors["spectype"].astype(str)))


def summarize_wmse(selection):
    values = wmse_test[selection]
    finite = np.isfinite(values)
    weight_sum = test_errors["sqrt_weight_sum"][selection].sum()
    weighted_sse = 2.0 * test_errors["reconstruction_term"][selection].sum()
    p16, median, p84 = (np.percentile(values[finite], [16, 50, 84])
                       if finite.any() else [np.nan] * 3)
    return dict(
        n_spectra=int(selection.sum()), n_scored=int(finite.sum()),
        n_undefined=int((~finite).sum()),
        mean_wmse=float(values[finite].mean()) if finite.any() else np.nan,
        median_wmse=float(median), p16_wmse=float(p16), p84_wmse=float(p84),
        pooled_wmse=float(weighted_sse / weight_sum) if weight_sum > 0 else np.nan,
    )


wmse_summary = {"ALL": summarize_wmse(np.ones(len(wmse_test), dtype=bool))}
wmse_summary.update({str(kind): summarize_wmse(spectype_test == kind)
                     for kind in np.unique(spectype_test)})
for kind, stats in wmse_summary.items():
    print(f"{kind}: {stats['n_scored']:,}/{stats['n_spectra']:,} scored, "
          f"mean WMSE={stats['mean_wmse']:.8g}, "
          f"median={stats['median_wmse']:.8g}, "
          f"p16–p84=[{stats['p16_wmse']:.8g}, {stats['p84_wmse']:.8g}], "
          f"pooled WMSE={stats['pooled_wmse']:.8g}")

from pathlib import Path
import csv
output_dir = Path("/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results/")
np.savez_compressed(output_dir / "dr2_test_recon_errors.npz", **test_errors)
with (output_dir / "dr2_test_wmse.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["TARGETID", "spectype", "wmse", "n_valid", "sqrt_weight_sum"])
    writer.writerows(zip(test_errors["TARGETID"], spectype_test, wmse_test,
                         test_errors["n_valid"], test_errors["sqrt_weight_sum"]))
with (output_dir / "dr2_test_wmse_summary.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=["spectype", *wmse_summary["ALL"]])
    writer.writeheader()
    writer.writerows(dict(spectype=kind, **stats) for kind, stats in wmse_summary.items())
print(f"Saved per-spectrum errors and WMSE summaries in {output_dir.resolve()}")

In [ ]:
if not hasattr(trans_loader.dataset, "M"):
    with np.load(trans_npz, allow_pickle=False) as data:
        transient_mask = data["M"].astype(bool) if "M" in data.files else None
    if transient_mask is not None and transient_mask.shape != trans_loader.dataset.X.shape:
        raise ValueError("NPZ mask does not match the existing loader; recreate trans_loader.")
    trans_loader.dataset.M = transient_mask

class TransientReconstructionDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        if dataset.W is None:
            raise ValueError("Transient WMSE requires W in the NPZ; unit weights are not substituted.")
        if dataset.X.shape != dataset.W.shape:
            raise ValueError("Transient X and W shapes must match.")
        if dataset.M is not None and dataset.M.shape != dataset.X.shape:
            raise ValueError("Transient M and X shapes must match.")
        if dataset.TARGETID is not None and len(dataset.TARGETID) != len(dataset):
            raise ValueError("TARGETID length must match X.")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, i):
        d = self.dataset
        x = torch.as_tensor(d.X[i])
        w = torch.as_tensor(d.W[i])
        if x.ndim == 1:
            x, w = x.unsqueeze(0), w.unsqueeze(0)
        if x.ndim != 2 or x.shape[0] != 1:
            raise ValueError("Transient spectra must have shape (L,) or (1,L).")
        if d.M is None:
            # Missing explicit mask: zero/negative weights carry no usable information.
            # Reject non-finite weights instead of silently dropping them.
            if not torch.isfinite(w).all():
                raise ValueError("Non-finite transient weights; check preprocessing.")
            mask = w > 0
        else:
            mask = torch.as_tensor(d.M[i], dtype=torch.bool).reshape(x.shape)
        return dict(x=x, weight=w, mask=mask,
                    targetid=int(d.TARGETID[i]) if d.TARGETID is not None else -1,
                    spectype="TRANSIENT")


trans_recon_dataset = TransientReconstructionDataset(trans_loader.dataset)
trans_recon_loader = DataLoader(trans_recon_dataset, batch_size=32, shuffle=False,
                                drop_last=False, num_workers=0)
print("Transient valid-pixel mask:",
      "M from NPZ" if trans_loader.dataset.M is not None else "W > 0 (M unavailable)")
transient_errors, transient_error_summary = evaluate_test_reconstruction(model, trans_recon_loader)
# Sequential loader: source_row provides unambiguous provenance even with repeated/missing IDs.
transient_errors["source_row"] = np.arange(len(trans_recon_dataset), dtype=np.int64)
wmse_transient = transient_errors["sqrt_weighted_mse"]
finite = np.isfinite(wmse_transient)
quantiles = np.percentile(wmse_transient[finite], [16, 50, 84]) if finite.any() else [np.nan] * 3
transient_error_summary.update(
    n_scored_wmse=int(finite.sum()),
    mean_per_spectrum_wmse=float(wmse_transient[finite].mean()) if finite.any() else np.nan,
    p16_per_spectrum_wmse=float(quantiles[0]),
    median_per_spectrum_wmse=float(quantiles[1]),
    p84_per_spectrum_wmse=float(quantiles[2]),
)
print("Transient reconstruction errors (sqrt_weighted_mse = pooled WMSE):")
for name, value in transient_error_summary.items():
    print(f"{name}: {value:,}" if isinstance(value, int) else f"{name}: {value:.8g}")

from pathlib import Path
import csv
transient_output_dir = Path("/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results/")
np.savez_compressed(transient_output_dir / "transient_reconstruction_errors.npz", **transient_errors)
with (transient_output_dir / "transient_reconstruction_errors.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["source_row", "TARGETID", "wmse", "mse", "rmse", "n_valid", "sqrt_weight_sum"])
    writer.writerows(zip(transient_errors["source_row"], transient_errors["TARGETID"],
                         wmse_transient, transient_errors["mse"], transient_errors["rmse"],
                         transient_errors["n_valid"], transient_errors["sqrt_weight_sum"]))
with (transient_output_dir / "transient_reconstruction_summary.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(transient_error_summary))
    writer.writeheader()
    writer.writerow(transient_error_summary)
print(f"Saved transient results in {transient_output_dir.resolve()}")

In [ ]:
from pathlib import Path
import numpy as np

test_dir = Path("/global/cfs/cdirs/desi/users/zixinq/vae_dr2/results")
transient_dir = test_dir  # Change if saved elsewhere

with np.load(
    test_dir / "dr2_test_recon_errors.npz",
    allow_pickle=False,
) as data:
    test_errors = {key: data[key] for key in data.files}

with np.load(
    transient_dir / "transient_reconstruction_errors.npz",
    allow_pickle=False,
) as data:
    transient_errors = {key: data[key] for key in data.files}

In [ ]:
"""Plot saved or in-memory per-spectrum reconstruction errors by spectral type."""
import numpy as np


def plot_reconstruction_by_type(test_errors, transient_errors,
                                metric="sqrt_weighted_mse"):
    import matplotlib.pyplot as plt

    labels = {"sqrt_weighted_mse": "WMSE",
              "mse": "MSE", "rmse": "RMSE"}
    if metric not in labels:
        raise ValueError(f"Choose one of {list(labels)}")
    types = np.char.upper(np.char.strip(np.asarray(test_errors["spectype"]).astype(str)))
    test_values = np.asarray(test_errors[metric], dtype=float)
    transient_values = np.asarray(transient_errors[metric], dtype=float)
    if types.ndim != 1 or test_values.shape != types.shape or transient_values.ndim != 1:
        raise ValueError("Expected aligned 1D type/score arrays and 1D transient scores.")
    groups = [(name, test_values[types == key], color, style)
              for key, name, color, style in [
                  ("GALAXY", "Galaxies", "#0072B2", "-"),
                  ("QSO", "QSOs", "#009E73", ":"),
                  ("STAR", "Stars", "#AF58BA", "-.")]]
    groups.append(("Transients", transient_values, "#FDBD3C", "-"))
    prepared, summary = [], []
    for name, raw, color, style in groups:
        finite = np.isfinite(raw)
        if np.any(raw[finite] < 0):
            raise ValueError(f"{name}: reconstruction errors cannot be negative.")
        values = raw[finite]
        quantiles = np.percentile(values, [16, 50, 84]) if values.size else [np.nan] * 3
        summary.append(dict(type=name, total=len(raw), scored=len(values),
                            undefined=int((~finite).sum()), zeros=int((values == 0).sum()),
                            mean=float(values.mean()) if values.size else np.nan,
                            p16=float(quantiles[0]), median=float(quantiles[1]), p84=float(quantiles[2])))
        if values.size:
            prepared.append((name, values, color, style))
    if not prepared:
        raise ValueError("No finite scores to plot. Run the reconstruction evaluations first.")
    all_values = np.concatenate([g[1] for g in prepared])
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, values, color, style in prepared:
        ax.hist(values, bins=100, density=True,
                histtype="bar", alpha=0.5, linewidth=0.7,
                color=color, edgecolor=color, linestyle=style,
                label=f"{name} (n={len(values):,})")
    ax.set(xlabel=f"{labels[metric]}",
           ylabel="Probability density per unit error",
           title="VAE Reconstruction Error by Spectral Type")
    ax.legend(frameon=False)
    ax.grid(alpha=0.18)
    ax.set_xlim(0, 0.007)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(rect=(0, 0.09, 1, 1))
    print("type           total   scored  undefined   zeros       mean     median        p16        p84")
    for row in summary:
        print(f"{row['type']:12s} {row['total']:7,d} {row['scored']:8,d} {row['undefined']:9,d} "
              f"{row['zeros']:7,d} {row['mean']:10.4g} {row['median']:10.4g} "
              f"{row['p16']:10.4g} {row['p84']:10.4g}")
    return fig, summary


In [ ]:
# Run both reconstruction evaluations above first, or load their saved NPZ files:
# with np.load("dr2_test_reconstruction_errors.npz", allow_pickle=False) as data:
#     test_errors = {key: data[key] for key in data.files}
# with np.load("transient_reconstruction_errors.npz", allow_pickle=False) as data:
#     transient_errors = {key: data[key] for key in data.files}

import matplotlib.pyplot as plt
reconstruction_fig, reconstruction_type_summary = plot_reconstruction_by_type(
    test_errors, transient_errors, metric="sqrt_weighted_mse", 
)
plt.show()

#reconstruction_fig.savefig("reconstruction_error_by_type.pdf", bbox_inches="tight")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path

# Use the calculated errors or load the saved NPZ files as shown above.
# Independent of the plot: this table always summarizes square-root-weighted MSE.
metric = "sqrt_weighted_mse"
types = np.char.upper(np.char.strip(np.asarray(test_errors["spectype"]).astype(str)))
values = np.asarray(test_errors[metric], dtype=float)
if types.ndim != 1 or values.shape != types.shape:
    raise ValueError("Test scores and spectral types must be aligned 1D arrays.")
score_groups = [(label, values[types == kind]) for kind, label in
                [("GALAXY", "Galaxies"), ("QSO", "QSOs"), ("STAR", "Stars")]]
score_groups.append(("Transients", np.asarray(transient_errors[metric], dtype=float)))
unknown = sorted(set(types) - {"GALAXY", "QSO", "STAR"})
if unknown:
    print("Test spectral types not included in this table:", unknown)
rows = []
for label, scores in score_groups:
    if scores.ndim != 1:
        raise ValueError(f"{label}: scores must be a 1D array.")
    finite = np.isfinite(scores)
    usable = scores[finite]
    if np.any(usable < 0):
        raise ValueError(f"{label}: WMSE cannot be negative.")
    median = float(np.median(usable)) if len(usable) else np.nan
    rows.append({"Spectral type": label, "Count": len(usable),
                 "Mean WMSE": float(usable.mean()) if len(usable) else np.nan,
                 "Median WMSE": median})
wmse_table = pd.DataFrame(rows)[
    ["Spectral type", "Count", "Median WMSE", "Mean WMSE"]
]
display(wmse_table.style.hide(axis="index").format({
    "Count": "{:,.0f}",
    **{column: "{:.4g}" for column in
       ["Median WMSE", "Mean WMSE"]},
}, na_rep="—").set_caption("VAE Reconstruction Error by Spectral Type"))

# Save numeric values at full precision, independently of display rounding.
wmse_table_path = Path("wmse_by_spectral_type.csv")
wmse_table.to_csv(wmse_table_path, index=False)
print(f"Saved table to {wmse_table_path.resolve()}")

In [ ]:
# Run the WMSE summary-table cell first.
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

pdf_path = Path("wmse_by_spectral_type.pdf")  # Or an absolute output path.
pdf_path.parent.mkdir(parents=True, exist_ok=True)
columns = ["Spectral type", "Count", "Median WMSE", "Mean WMSE"]
pdf_values = wmse_table[columns].copy()
pdf_values["Count"] = pdf_values["Count"].map(lambda x: f"{x:,.0f}")
for column in ["Median WMSE", "Mean WMSE"]:
    pdf_values[column] = pdf_values[column].map(
        lambda x: "-" if pd.isna(x) else f"{x:.4g}"
    )

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.axis("off")
fig.suptitle("VAE Reconstruction Error by Spectral Type", fontsize=13, y=0.96)
artist = ax.table(cellText=pdf_values.values, colLabels=columns,
                  cellLoc="right", colLoc="right",
                  colWidths=[0.27, 0.23, 0.25, 0.25], bbox=[0, 0.12, 1, 0.80])
artist.auto_set_font_size(False)
artist.set_fontsize(10)
for (row, col), cell in artist.get_celld().items():
    cell.set_edgecolor("#D5DAE0")
    cell.set_linewidth(0.5)
    cell.PAD = 0.10
    if row == 0:
        cell.set_facecolor("#EAF0F6")
        cell.set_text_props(weight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#F7F9FB")
    if col == 0:
        cell.set_text_props(ha="left")

fig.subplots_adjust(left=0.03, right=0.97, bottom=0.16, top=0.86)
fig.savefig(pdf_path, format="pdf", bbox_inches="tight", pad_inches=0.15)
plt.show()
plt.close(fig)
print(f"Saved PDF to {pdf_path.resolve()}")

In [ ]:
#classifier

import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

X_real = np.vstack([
    mu_galaxy,
    mu_qso,
    mu_star,
    mu_tr,
])

labels_real = np.concatenate([
    np.full(len(mu_galaxy), "Galaxy"),
    np.full(len(mu_qso), "QSO"),
    np.full(len(mu_star), "STAR"),
    np.full(len(mu_tr), "Transient"),
])

tid_real = np.concatenate([
    tid_galaxy,
    tid_qso,
    tid_star,
    tid_tr,
])

print("X:", X_real.shape)
print("labels:", labels_real.shape)
print("TARGETIDs:", tid_real.shape)

unique, counts = np.unique(labels_real, return_counts=True)

for name, count in zip(unique, counts):
    print(name, count)

le = LabelEncoder()

y_real = le.fit_transform(labels_real)

print(le.classes_)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

tr_ids = np.asarray(tid_tr).reshape(-1)

tr_train_ids, tr_test_ids = train_test_split(
    np.unique(tr_ids),
    test_size=0.2,
    random_state=42,
)

tr_train_mask = np.isin(tr_ids, tr_train_ids)
tr_test_mask = np.isin(tr_ids, tr_test_ids)


# Assemble features, labels, and IDs in the same order.
def assemble_split(groups):
    features, labels, targetids = [], [], []

    for name, mu, tids in groups:
        mu = np.asarray(mu)
        tids = np.asarray(tids).reshape(-1)

        if mu.ndim != 2 or len(mu) != len(tids):
            raise ValueError(
                f"{name}: incompatible shapes {mu.shape}, {tids.shape}"
            )

        features.append(mu)
        labels.append(np.full(len(mu), name))
        targetids.append(tids)

    return (
        np.vstack(features).astype(np.float32, copy=False),
        np.concatenate(labels),
        np.concatenate(targetids),
    )


X_train_real, labels_train_real, tid_train_real = assemble_split([
    ("Galaxy", mu_galaxy, tid_galaxy),
    ("QSO", mu_qso, tid_qso),
    ("STAR", mu_star, tid_star),
    ("Transient", mu_tr[tr_train_mask], tr_ids[tr_train_mask]),
])

X_test, labels_test, tid_test = assemble_split([
    ("Galaxy", mu_galaxy_test, tid_galaxy_test),
    ("QSO", mu_qso_test, tid_qso_test),
    ("STAR", mu_star_test, tid_star_test),
    ("Transient", mu_tr[tr_test_mask], tr_ids[tr_test_mask]),
])

assert X_train_real.shape[1] == X_test.shape[1]

le = LabelEncoder()
y_train_real = le.fit_transform(labels_train_real)
y_test = le.transform(labels_test)

sample_weight_real = compute_sample_weight(
    class_weight="balanced",
    y=y_train_real,
)

print("Class order:", le.classes_)

for name, X, y in [
    ("Train", X_train_real, y_train_real),
    ("Test", X_test, y_test),
]:
    print(f"\n{name}: {X.shape}")
    for cls, count in zip(
        le.classes_,
        np.bincount(y, minlength=len(le.classes_)),
    ):
        print(f"  {cls}: {count:,}")
        

from xgboost import XGBClassifier
from sklearn.metrics import classification_report

clf_real = XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    reg_alpha=0.0,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=8,
)

clf_real.fit(
    X_train_real,
    y_train_real,
    sample_weight=sample_weight_real,
)

y_pred_real = clf_real.predict(X_test)
y_prob_real = clf_real.predict_proba(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    y_test,
    y_pred_real,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    digits=4,
    zero_division=0,
))

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_real,
    display_labels=le.classes_,
    normalize="true",
    values_format=".2f",
)

plt.title("Real-only training")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
)

# 1. Label simulated transients using the existing encoder.
transient_label = int(le.transform(["Transient"])[0])

mu_sim_train = np.asarray(mu_sim, dtype=np.float32)
tid_sim_train = np.asarray(tid_sim).reshape(-1)

assert mu_sim_train.ndim == 2
assert mu_sim_train.shape[1] == X_train_real.shape[1]
assert len(mu_sim_train) == len(tid_sim_train)

y_sim = np.full(
    len(mu_sim_train),
    transient_label,
    dtype=y_train_real.dtype,
)

# 2. Augment only the training set.
X_train_aug = np.vstack([
    X_train_real,
    mu_sim_train,
])

y_train_aug = np.concatenate([
    y_train_real,
    y_sim,
])

# Preserve the simulation TARGETIDs loaded from your file.
# Do not overwrite tid_sim with np.arange().
tid_train_aug = np.concatenate([
    tid_train_real,
    tid_sim_train,
])

# Keep source information because real and simulated IDs may overlap.
source_train_aug = np.concatenate([
    np.full(len(y_train_real), "real"),
    np.full(len(y_sim), "simulation"),
])

print("Real-only training:", X_train_real.shape)
print("Augmented training:", X_train_aug.shape)
print("Unchanged real test set:", X_test.shape)

for label, count in zip(
    *np.unique(y_train_aug, return_counts=True)
):
    print(f"{le.inverse_transform([label])[0]}: {count:,}")

# 3. Recompute class weights for the augmented training population.
sample_weight_aug = compute_sample_weight(
    class_weight="balanced",
    y=y_train_aug,
)

# 4. Use the same hyperparameters as the real-only classifier.
# clone() returns a new, unfitted estimator.
clf_aug = clone(clf_real)

clf_aug.fit(
    X_train_aug,
    y_train_aug,
    sample_weight=sample_weight_aug,
)

# 5. Evaluate on exactly the same real test set.
y_pred_aug = clf_aug.predict(X_test)
y_prob_aug = clf_aug.predict_proba(X_test)

assert np.array_equal(clf_aug.classes_, clf_real.classes_)

print(classification_report(
    y_test,
    y_pred_aug,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    digits=4,
    zero_division=0,
))

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_aug,
    labels=np.arange(len(le.classes_)),
    display_labels=le.classes_,
    normalize="true",
    values_format=".2f",
    cmap="Blues",
    ax=ax,
)

ax.set_title("Real + simulated transient training")
fig.tight_layout()
plt.show()

In [ ]:
def as_spectra_2d(name, arr):
    arr = np.asarray(arr)
    print(f"{name}: before {arr.shape}")

    if arr.ndim == 3 and arr.shape[1] == 1:      # (N, 1, L)
        arr = arr[:, 0, :]
    elif arr.ndim == 3 and arr.shape[2] == 1:    # (N, L, 1)
        arr = arr[:, :, 0]

    if arr.ndim != 2:
        raise ValueError(f"{name} should reduce to (N, L), got {arr.shape}")

    print(f"{name}: after  {arr.shape}")
    return arr

flux_galaxy_2d = as_spectra_2d("flux_galaxy", flux_galaxy)
flux_qso_2d    = as_spectra_2d("flux_qso", flux_qso)
flux_star_2d    = as_spectra_2d("flux_star", flux_star)
flux_tr_2d     = as_spectra_2d("flux_tr", flux_tr)

# Check row counts and wavelength-bin counts before combining.
assert len(flux_galaxy_2d) == len(mu_galaxy)
assert len(flux_qso_2d) == len(mu_qso)
assert len(flux_tr_2d) == len(mu_tr)
assert flux_galaxy_2d.shape[1] == flux_qso_2d.shape[1] == flux_tr_2d.shape[1]

flux_real = np.vstack([flux_galaxy_2d, flux_qso_2d, flux_star_2d, flux_tr_2d])
print("flux_real:", flux_real.shape)

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

_, _, _, _, _, tid_test_check, _, flux_test = train_test_split(
    X_real,
    y_real,
    tid_real,
    flux_real,
    test_size=0.2,
    random_state=42,
    stratify=y_real,
)

assert np.array_equal(tid_test, tid_test_check)
print("flux_test:", flux_test.shape)

In [ ]:
real_wrong = np.flatnonzero(y_pred_real != y_test)
aug_wrong  = np.flatnonzero(y_pred_aug  != y_test)

both_wrong      = np.intersect1d(real_wrong, aug_wrong)
fixed_by_aug    = np.setdiff1d(real_wrong, aug_wrong)
wrong_only_aug  = np.setdiff1d(aug_wrong, real_wrong)

print("Both wrong:", len(both_wrong))
print("Fixed by augmented model:", len(fixed_by_aug))
print("Wrong only for augmented model:", len(wrong_only_aug))

In [ ]:
import pandas as pd 

error_positions = np.union1d(real_wrong, aug_wrong)

errors = pd.DataFrame({
    "test_position": error_positions,
    "TARGETID": tid_test[error_positions],
    "true": le.inverse_transform(y_test[error_positions]),
    "real_only_pred": le.inverse_transform(y_pred_real[error_positions]),
    "augmented_pred": le.inverse_transform(y_pred_aug[error_positions]),
})

errors.head(30)

In [ ]:
def plot_ranked_mistakes(
    y_pred, y_prob, model_name,
    n_per_class=20, highest_first=True, wave=None,
    ncols=5,
):
    class_names = ["Transient", "Galaxy", "QSO"]
    class_ids = le.transform(class_names)
    y_pred = np.asarray(y_pred, dtype=int)

    confidence = y_prob[np.arange(len(y_pred)), y_pred]
    x = np.arange(flux_test.shape[1]) if wave is None else np.asarray(wave)

    rows_per_class = int(np.ceil(n_per_class / ncols))
    nrows = len(class_names) * rows_per_class

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4 * ncols, 3.3 * nrows),
        squeeze=False,
    )

    selected = {}

    for group, (true_name, class_id) in enumerate(zip(class_names, class_ids)):
        mistakes = np.flatnonzero(
            (y_test == class_id) & (y_pred != y_test)
        )

        order = np.argsort(confidence[mistakes])
        if highest_first:
            order = order[::-1]

        chosen = mistakes[order[:n_per_class]]
        selected[true_name] = chosen

        print(f"{model_name} — {true_name}: {len(mistakes)} errors")

        class_axes = axes[
            group * rows_per_class : (group + 1) * rows_per_class
        ].ravel()

        for j, ax in enumerate(class_axes):
            if j >= len(chosen):
                ax.axis("off")
                continue

            pos = chosen[j]
            pred_name = le.classes_[y_pred[pos]]

            ax.plot(x, flux_test[pos], lw=0.8)
            ax.set_title(
                f"Rank {j + 1} | ID {tid_test[pos]}\n"
                f"{true_name} → {pred_name}\n"
                f"P(pred)={confidence[pos]:.3f} | "
                f"P(true)={y_prob[pos, class_id]:.3f}",
                fontsize=10,
            )
            ax.set_xlabel("Pixel" if wave is None else "Wavelength", fontsize=9)
            ax.set_ylabel("Flux", fontsize=9)
            ax.tick_params(labelsize=8)

    direction = "highest" if highest_first else "lowest"
    fig.suptitle(
        f"{model_name}: errors ranked by {direction} predicted probability",
        fontsize=16,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

    return selected

In [ ]:
ranked_real = plot_ranked_mistakes(
    y_pred_real, y_prob_real, "Real-only model",
    n_per_class=20,
)

ranked_aug = plot_ranked_mistakes(
    y_pred_aug, y_prob_aug, "Real + simulated model",
    n_per_class=20,
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    precision_recall_fscore_support,
    average_precision_score,
    accuracy_score,
    f1_score,
)

out_dir = Path("classifier_results")
out_dir.mkdir(parents=True, exist_ok=True)

class_names = np.asarray(le.classes_)
class_ids = np.arange(len(class_names))
transient_label = int(le.transform(["Transient"])[0])

y_true = np.asarray(y_test)
true_transient = y_true == transient_label

models = {
    "Real only": {
        "pred": np.asarray(y_pred_real),
        "prob": np.asarray(y_prob_real),
    },
    "Real + simulated": {
        "pred": np.asarray(y_pred_aug),
        "prob": np.asarray(y_prob_aug),
    },
}

# Both models must use the same label encoding.
assert np.array_equal(clf_real.classes_, class_ids)
assert np.array_equal(clf_aug.classes_, class_ids)

for name, result in models.items():
    assert result["pred"].shape == y_true.shape, name
    assert result["prob"].shape == (
        len(y_true), len(class_names)
    ), name
    assert np.isfinite(result["prob"]).all(), name

assert 0 < true_transient.sum() < len(y_true)

# ------------------------------------------------------------
# 1. Dataset composition
# ------------------------------------------------------------
dataset_table = pd.DataFrame(
    {
        "Real-only training": np.bincount(
            y_train_real, minlength=len(class_names)
        ),
        "Augmented training": np.bincount(
            y_train_aug, minlength=len(class_names)
        ),
        "Real test": np.bincount(
            y_true, minlength=len(class_names)
        ),
    },
    index=pd.Index(class_names, name="Class"),
)

dataset_table["Simulated training"] = (
    dataset_table["Augmented training"]
    - dataset_table["Real-only training"]
)

dataset_table = dataset_table[
    [
        "Real-only training",
        "Simulated training",
        "Augmented training",
        "Real test",
    ]
]

# ------------------------------------------------------------
# 2. Per-class performance and overall comparison
# ------------------------------------------------------------
class_rows = []
summary = {}
candidate_rows = []

for name, result in models.items():
    pred = result["pred"]
    prob = result["prob"]

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            pred,
            labels=class_ids,
            zero_division=0,
        )
    )

    for i, cls in enumerate(class_names):
        class_rows.append({
            "Class": cls,
            "Model": name,
            "Precision": precision[i],
            "Recall": recall[i],
            "F1": f1[i],
            "Test support": int(support[i]),
        })

    summary[name] = {
        "Transient precision": precision[transient_label],
        "Transient recall": recall[transient_label],
        "Transient F1": f1[transient_label],
        "Transient AP": average_precision_score(
            true_transient,
            prob[:, transient_label],
        ),
        "Macro-F1": f1_score(
            y_true,
            pred,
            labels=class_ids,
            average="macro",
            zero_division=0,
        ),
        "Accuracy": accuracy_score(y_true, pred),
    }

    # Candidate selection uses the multiclass predicted label.
    selected = pred == transient_label

    tp = int(np.sum(selected & true_transient))
    fp = int(np.sum(selected & ~true_transient))
    fn = int(np.sum(~selected & true_transient))
    tn = int(np.sum(~selected & ~true_transient))

    candidate_rows.append({
        "Model": name,
        "Selected candidates": tp + fp,
        "Recovered transients": tp,
        "Missed transients": fn,
        "False positives": fp,
        "Contamination": fp / (tp + fp) if tp + fp else np.nan,
        "FP per 10,000 non-transients": 10000 * fp / (fp + tn),
    })

per_class_table = pd.DataFrame(class_rows).set_index(
    ["Class", "Model"]
)

# Side-by-side layout for the paper.
paper_class_table = per_class_table[
    ["Precision", "Recall", "F1"]
].unstack("Model")

summary_table = pd.DataFrame(summary)
summary_table.index.name = "Metric"

# All summary metrics are fractions, so multiply differences
# by 100 to express them as percentage-point changes.
summary_table["Change (percentage points)"] = 100 * (
    summary_table["Real + simulated"]
    - summary_table["Real only"]
)

candidate_table = pd.DataFrame(candidate_rows).set_index("Model")

# ------------------------------------------------------------
# 3. Display
# ------------------------------------------------------------
print("DATASET COMPOSITION")
display(dataset_table)

print("PER-CLASS PERFORMANCE")
display(paper_class_table.round(4))

print("MODEL COMPARISON")
display(summary_table.round(4))

print("TRANSIENT CANDIDATES — MULTICLASS ARGMAX")
display(candidate_table.round(4))

# ------------------------------------------------------------
# 4. Export CSV and LaTeX
# ------------------------------------------------------------
tables = {
    "dataset_composition": dataset_table,
    "per_class_performance": per_class_table,
    "per_class_comparison": paper_class_table,
    "model_comparison": summary_table,
    "transient_candidates": candidate_table,
}

for filename, table in tables.items():
    table.to_csv(out_dir / f"{filename}.csv")
    table.to_latex(
        out_dir / f"{filename}.tex",
        float_format="%.4f",
        escape=True,
        na_rep="—",
    )

print(f"Reports saved to: {out_dir.resolve()}")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import PercentFormatter
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
)
from IPython.display import display

out_dir = Path("classifier_paper_results")
out_dir.mkdir(parents=True, exist_ok=True)

class_names = np.asarray(le.classes_)
class_ids = np.arange(len(class_names))
tr_idx = int(le.transform(["Transient"])[0])

y_true = np.asarray(y_test)
test_ids = np.asarray(tid_test).reshape(-1)
is_tr = y_true == tr_idx

models = {
    "Real only": {
        "pred": np.asarray(y_pred_real),
        "prob": np.asarray(y_prob_real),
        "color": "#226E9C",
        "linestyle": "-",
    },
    "Real + simulated": {
        "pred": np.asarray(y_pred_aug),
        "prob": np.asarray(y_prob_aug),
        "color": "#C44E52",
        "linestyle": "--",
    },
}

assert np.array_equal(clf_real.classes_, class_ids)
assert np.array_equal(clf_aug.classes_, class_ids)
assert len(test_ids) == len(y_true)
assert 0 < is_tr.sum() < len(y_true)

for name, result in models.items():
    assert result["pred"].shape == y_true.shape, name
    assert result["prob"].shape == (len(y_true), len(class_names)), name
    assert np.isfinite(result["prob"]).all(), name

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def savefig(fig, name):
    fig.savefig(out_dir / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(
        out_dir / f"{name}.png",
        dpi=300, bbox_inches="tight",
    )

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(12, 5), constrained_layout=True
)

for ax, (name, result) in zip(axes, models.items()):
    cm = confusion_matrix(
        y_true, result["pred"], labels=class_ids
    )

    totals = cm.sum(axis=1, keepdims=True)
    fractions = np.divide(
        cm,
        totals,
        out=np.zeros_like(cm, dtype=float),
        where=totals != 0,
    )

    im = ax.imshow(
        fractions, cmap="Blues", vmin=0, vmax=1
    )

    for i in class_ids:
        for j in class_ids:
            percentage = (
                f"{fractions[i, j]:.1%}"
                if totals[i, 0] else "N/A"
            )
            ax.text(
                j, i,
                f"{percentage}\n({cm[i, j]:,})",
                ha="center", va="center",
                color="white" if fractions[i, j] > 0.55 else "black",
                fontsize=10,
            )

    ax.set(
        xticks=class_ids,
        yticks=class_ids,
        xticklabels=class_names,
        yticklabels=class_names,
        xlabel="Predicted class",
        ylabel="True class",
        title=name,
    )

cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.85)
cbar.set_label("Fraction within true class")
cbar.ax.yaxis.set_major_formatter(PercentFormatter(1))

#savefig(fig, "confusion_matrices")
plt.show()

In [ ]:
thresholds = {
    "Real only": None,
    "Real + simulated": None,
}

operating_rows = []

for name, result in models.items():
    threshold = thresholds[name]

    if threshold is None:
        selected = result["pred"] == tr_idx
        rule = "Multiclass argmax"
    else:
        threshold = float(threshold)
        if not 0 <= threshold <= 1:
            raise ValueError("Threshold must be between 0 and 1.")

        selected = result["prob"][:, tr_idx] >= threshold
        rule = f"P(Transient) >= {threshold:.4f}"

    result["selected"] = selected
    result["selection_rule"] = rule

    tp = int(np.sum(selected & is_tr))
    fp = int(np.sum(selected & ~is_tr))
    fn = int(np.sum(~selected & is_tr))
    tn = int(np.sum(~selected & ~is_tr))

    purity = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn)

    result["operating_precision"] = purity
    result["operating_recall"] = recall

    operating_rows.append({
        "Model": name,
        "Rule": rule,
        "Candidates": tp + fp,
        "Recovered transients": tp,
        "Missed transients": fn,
        "False positives": fp,
        "Precision": purity,
        "Recall": recall,
        "Contamination": 1 - purity,
        "FP per 10,000 non-transients": 10000 * fp / (fp + tn),
    })

operating_table = pd.DataFrame(operating_rows).set_index("Model")

display(operating_table.round(4))
operating_table.to_csv(out_dir / "operating_points.csv")
operating_table.to_latex(
    out_dir / "operating_points.tex",
    float_format="%.4f",
    escape=True,
    na_rep="—",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)

tr_idx = int(le.transform(["Transient"])[0])
y_binary = (np.asarray(y_test) == tr_idx).astype(int)

model_scores = {
    "Real only": np.asarray(y_prob_real)[:, tr_idx],
    "Real + simulated": np.asarray(y_prob_aug)[:, tr_idx],
}

colors = ["#226E9C", "#C44E52"]
styles = ["-", "--"]

fig, (ax_roc, ax_pr) = plt.subplots(
    1, 2, figsize=(12, 5), constrained_layout=True
)

for (name, scores), color, style in zip(
    model_scores.items(), colors, styles
):
    # ROC
    fpr, tpr, _ = roc_curve(y_binary, scores)
    roc_auc = roc_auc_score(y_binary, scores)

    ax_roc.plot(
        fpr, tpr,
        color=color,
        linestyle=style,
        linewidth=2,
        label=f"{name} (AUC = {roc_auc:.4f})",
    )

    # Precision–recall
    precision, recall, _ = precision_recall_curve(
        y_binary, scores
    )
    ap = average_precision_score(y_binary, scores)

    ax_pr.step(
        recall, precision,
        where="post",
        color=color,
        linestyle=style,
        linewidth=2,
        label=f"{name} (AP = {ap:.4f})",
    )

ax_roc.plot(
    [0, 1], [0, 1],
    color="0.5", linestyle=":", label="Random ranking"
)
ax_roc.set(
    xlabel="False positive rate",
    ylabel="True positive rate (recall)",
    title="Transient ROC",
    xlim=(0, 1),
    ylim=(0, 1.02),
)

prevalence = y_binary.mean()
ax_pr.axhline(
    prevalence,
    color="0.5",
    linestyle=":",
    label=f"Transient fraction = {prevalence:.3e}",
)
ax_pr.set(
    xlabel="Recall (completeness)",
    ylabel="Precision (purity)",
    title="Transient precision–recall",
    xlim=(0, 1),
    ylim=(0, 1.02),
)

for ax in (ax_roc, ax_pr):
    ax.grid(alpha=0.2)
    ax.legend(loc="best", fontsize=9)

#fig.savefig("roc_pr_comparison.pdf", bbox_inches="tight")
#fig.savefig("roc_pr_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
tr_test_rows = np.flatnonzero(tr_test_mask)

flux_blocks = [
    flux_galaxy_test,
    flux_qso_test,
    flux_star_test,
    flux_tr,
]

block_sizes = [
    len(flux_galaxy_test),
    len(flux_qso_test),
    len(flux_star_test),
    len(tr_test_rows),
]

block_ends = np.cumsum(block_sizes)
block_starts = np.r_[0, block_ends[:-1]]

# Verify that the flux blocks follow the test metadata order.
expected_ids = np.concatenate([
    np.asarray(tid_galaxy_test).reshape(-1),
    np.asarray(tid_qso_test).reshape(-1),
    np.asarray(tid_star_test).reshape(-1),
    np.asarray(tid_tr).reshape(-1)[tr_test_rows],
])

expected_labels = np.concatenate([
    np.full(block_sizes[0], le.transform(["Galaxy"])[0]),
    np.full(block_sizes[1], le.transform(["QSO"])[0]),
    np.full(block_sizes[2], le.transform(["STAR"])[0]),
    np.full(block_sizes[3], tr_idx),
])

assert block_ends[-1] == len(y_true)
assert np.array_equal(expected_ids, test_ids), "Test ID order differs."
assert np.array_equal(expected_labels, y_true), "Test label order differs."


def get_test_flux(index):
    block = int(np.searchsorted(block_ends, index, side="right"))
    local_index = index - block_starts[block]

    if block == 3:
        local_index = tr_test_rows[local_index]

    return np.asarray(flux_blocks[block][local_index]).reshape(-1)


def plot_examples(
    model_name,
    n_per=3,
    seed=42,
    wave=None,
    selection="random",
):
    """
    selection:
        'random'   — reproducible random examples
        'extreme'  — highest-score detections/false positives,
                     lowest-score missed transients

    wave must match the preprocessed spectral grid, in Angstrom.
    """
    if selection not in {"random", "extreme"}:
        raise ValueError("selection must be 'random' or 'extreme'.")

    result = models[model_name]
    selected = result["selected"]
    scores = result["prob"][:, tr_idx]

    categories = {
        "Recovered transient": is_tr & selected,
        "Missed transient": is_tr & ~selected,
        "False positive": ~is_tr & selected,
    }

    rng = np.random.default_rng(seed)

    fig, axes = plt.subplots(
        3, n_per,
        figsize=(4.5 * n_per, 9),
        squeeze=False,
        constrained_layout=True,
    )

    example_rows = []

    for row, (category, mask) in enumerate(categories.items()):
        pool = np.flatnonzero(mask)
        n = min(n_per, len(pool))

        if selection == "random":
            chosen = rng.choice(pool, size=n, replace=False)
        else:
            order = np.argsort(scores[pool])
            if category != "Missed transient":
                order = order[::-1]
            chosen = pool[order[:n]]

        for col, ax in enumerate(axes[row]):
            if col >= len(chosen):
                ax.axis("off")
                continue

            index = int(chosen[col])
            flux = get_test_flux(index)

            x = (
                np.arange(len(flux))
                if wave is None
                else np.asarray(wave)
            )
            assert x.shape == flux.shape

            ax.plot(
                x, flux,
                color=result["color"],
                linewidth=0.8,
            )

            true_class = class_names[y_true[index]]
            predicted_class = class_names[result["pred"][index]]

            ax.set_title(
                f"{category}\n"
                f"True: {true_class}; argmax: {predicted_class}\n"
                f"TARGETID: {test_ids[index]}\n"
                f"P(Transient) = {scores[index]:.3f}",
                fontsize=9,
            )
            ax.set_xlabel(
                "Spectral bin" if wave is None else "Wavelength (Å)"
            )
            ax.set_ylabel("Preprocessed flux")

            example_rows.append({
                "Model": model_name,
                "Category": category,
                "Selection": selection,
                "Test row": index,
                "TARGETID": test_ids[index],
                "True class": true_class,
                "Argmax class": predicted_class,
                "Transient score": scores[index],
            })

    fig.suptitle(
        f"{model_name} — {result['selection_rule']}\n"
        f"{selection.capitalize()} examples",
        fontsize=13,
    )

    slug = (
        model_name.lower()
        .replace(" + ", "_plus_")
        .replace(" ", "_")
    )
    filename = f"examples_{slug}_{selection}"

    savefig(fig, filename)
    pd.DataFrame(example_rows).to_csv(
        out_dir / f"{filename}.csv", index=False
    )
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = Path("classifier_paper_results")
out_dir.mkdir(parents=True, exist_ok=True)

class_names = np.asarray(le.classes_)
y_true = np.asarray(y_test)
test_ids = np.asarray(tid_test).reshape(-1)

class_colors = {
    "Galaxy": "#226E9C",
    "QSO": "#55A868",
    "STAR": "#AF58BA",
    "Transient": "#FDBD3C",
}


def plot_misclassified_pairs(
    y_pred,
    y_prob,
    model_name,
    n_per_pair=6,
    selection="random",
    seed=42,
    wave=None,
):
    """
    One figure per true class.
    Rows: incorrect predicted classes.
    Columns: example spectra.

    selection:
        "random"     — random examples from each confusion pair
        "confidence" — highest probability for the incorrect class
    """
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    assert y_pred.shape == y_true.shape
    assert y_prob.shape == (len(y_true), len(class_names))

    if selection not in {"random", "confidence"}:
        raise ValueError("Use 'random' or 'confidence'.")

    rng = np.random.default_rng(seed)

    for true_idx, true_name in enumerate(class_names):
        wrong_classes = [
            i for i in range(len(class_names))
            if i != true_idx
        ]

        fig, axes = plt.subplots(
            len(wrong_classes),
            n_per_pair,
            figsize=(3.5 * n_per_pair, 3.1 * len(wrong_classes)),
            squeeze=False,
            constrained_layout=True,
        )

        for row, pred_idx in enumerate(wrong_classes):
            pred_name = class_names[pred_idx]

            pool = np.flatnonzero(
                (y_true == true_idx) & (y_pred == pred_idx)
            )
            n_show = min(n_per_pair, len(pool))

            if selection == "random":
                chosen = rng.choice(
                    pool, size=n_show, replace=False
                )
            else:
                ranked = np.argsort(
                    -y_prob[pool, pred_idx],
                    kind="stable",
                )
                chosen = pool[ranked[:n_show]]


            for col, ax in enumerate(axes[row]):
                if col >= n_show:
                    ax.remove()
                    continue
            
                index = int(chosen[col])
                flux = get_test_flux(index)
            
                x = (
                    np.arange(len(flux))
                    if wave is None
                    else np.asarray(wave)
                )
                if x.shape != flux.shape:
                    raise ValueError("Wavelength grid must match the flux shape.")
            
                ax.plot(
                    x, flux,
                    color=class_colors.get(true_name, "#226E9C"),
                    linewidth=0.8,
                )
            
                ax.set_title(
                    f"TARGETID {test_ids[index]}\n"
                    f"P({pred_name}) = {y_prob[index, pred_idx]:.3f}",
                    fontsize=9,
                )
                ax.tick_params(labelsize=8)
                ax.set_xlabel(
                    "Spectral bin" if wave is None else "Wavelength (Å)",
                    fontsize=9,
                )
            
                if col == 0:
                    ax.set_ylabel(
                        f"{true_name} → {pred_name}\n"
                        f"N = {len(pool):,}\n"
                        "Preprocessed flux",
                        fontsize=10,
                    )

        fig.suptitle(
            f"{model_name}: misclassified {true_name} spectra\n"
            f"{'Random examples' if selection == 'random' else 'Highest-confidence errors'}",
            fontsize=14,
        )

        slug = model_name.lower().replace(" + ", "_plus_").replace(" ", "_")
        filename = f"{slug}_true_{true_name.lower()}_{selection}"
        plt.show()
        plt.close(fig)
'''
        fig.savefig(
            out_dir / f"{filename}.pdf",
            bbox_inches="tight",
        )
        fig.savefig(
            out_dir / f"{filename}.png",
            dpi=300,
            bbox_inches="tight",
        )
'''
        

In [ ]:
plot_misclassified_pairs(
    y_pred_real,
    y_prob_real,
    model_name="Real only",
    n_per_pair=6,
    selection="confidence",
)

plot_misclassified_pairs(
    y_pred_aug,
    y_prob_aug,
    model_name="Real + simulated",
    n_per_pair=6,
    selection="confidence",
)

In [ ]:
#plot reconstruction
"""Small, reproducible VAE reconstruction figures for the evaluation notebook."""
from itertools import islice
from pathlib import Path
import csv
import json
import numpy as np


def _numpy(value):
    return value.detach().cpu().numpy() if hasattr(value, 'detach') else np.asarray(value)


def _spectra(value):
    value = _numpy(value)
    if value.ndim == 3 and value.shape[1] == 1:
        value = value[:, 0]
    if value.ndim != 2:
        raise ValueError('Expected (N,L) or (N,1,L) spectra.')
    return value


def sample_examples(loader, kinds, source, pool_size=64, max_batches=200, seed=42):
    """Uniform reservoir per type within the scanned loader prefix, not full survey."""
    if pool_size < 1 or (max_batches is not None and max_batches < 1):
        raise ValueError('pool_size and max_batches must be positive.')
    rng = np.random.default_rng(seed)
    pools = {kind: [] for kind in kinds}
    counts = {kind: 0 for kind in kinds}
    offset = 0
    batches = loader if max_batches is None else islice(loader, max_batches)
    for batch in batches:
        x, w, mask = (_spectra(batch[key]) for key in ('x', 'weight', 'mask'))
        mask = mask.astype(bool)
        if x.shape != w.shape or x.shape != mask.shape:
            raise ValueError('Flux, weights and mask must align.')
        ids = _numpy(batch['targetid'])
        types = np.char.upper(np.char.strip(np.asarray(batch['spectype']).astype(str)))
        if ids.shape != (len(x),) or types.shape != (len(x),):
            raise ValueError('Metadata must align with spectra.')
        for i, kind in enumerate(types):
            if kind not in pools:
                continue
            if not np.isfinite(x[i]).all() or not np.isfinite(w[i][mask[i]]).all():
                raise ValueError(f'Nonfinite input at {source}, stream row {offset+i}.')
            if not np.any(mask[i] & (w[i] > 0)):
                continue
            counts[kind] += 1
            slot = len(pools[kind]) if len(pools[kind]) < pool_size else int(rng.integers(counts[kind]))
            if slot >= pool_size:
                continue
            row = dict(kind=kind, source=str(source), stream_row=offset+i,
                       targetid=int(ids[i]), x=x[i].copy(), w=w[i].copy(), mask=mask[i].copy())
            if slot == len(pools[kind]):
                pools[kind].append(row)
            else:
                pools[kind][slot] = row
        offset += len(x)
    return pools, dict(source=str(source), seed=seed, pool_size=pool_size,
                       max_batches=max_batches, scanned_rows=offset, eligible_by_type=counts)


def transient_example_batches(path, batch_size=256):
    """Sequential NPZ rows; stream_row is the original NPZ row index."""
    with np.load(path, allow_pickle=False) as data:
        x, w = _spectra(data['X']), _spectra(data['W'])
        mask = _spectra(data['M']).astype(bool) if 'M' in data else w > 0
        if 'M' not in data and not np.isfinite(w).all():
            raise ValueError('Nonfinite transient weights with no explicit mask.')
        ids = data['TARGETID'] if 'TARGETID' in data else np.full(len(x), -1, dtype=np.int64)
        if w.shape != x.shape or mask.shape != x.shape or ids.shape != (len(x),):
            raise ValueError('Transient arrays must align.')
        for start in range(0, len(x), batch_size):
            end = start + batch_size
            yield dict(x=x[start:end], weight=w[start:end], mask=mask[start:end],
                       targetid=ids[start:end], spectype=['TRANSIENT'] * len(x[start:end]))


def reconstruct_examples(model, pools, per_type=1):
    """Select examples nearest median WMSE in each sampled pool; no best-fit selection."""
    import torch
    if per_type < 1:
        raise ValueError('per_type must be positive.')
    parameter = next(model.parameters())
    was_training = model.training
    selected = []
    model.eval()
    try:
        with torch.inference_mode():
            for kind in ('GALAXY', 'QSO', 'STAR', 'TRANSIENT'):
                rows = pools.get(kind, [])
                if len(rows) < per_type:
                    raise ValueError(f'{kind}: found {len(rows)} candidates; increase scan or supply its dataset.')
                x = torch.as_tensor(np.stack([r['x'] for r in rows]),
                                    device=parameter.device, dtype=parameter.dtype).unsqueeze(1)
                mu, _ = model.encode(x)
                prediction = _spectra(model.decode(mu))
                if prediction.shape != (len(rows), x.shape[-1]) or not np.isfinite(prediction).all():
                    raise ValueError('Invalid decoder output.')
                for row, pred in zip(rows, prediction):
                    row['reconstruction'] = pred.copy()
                    weights = np.where(row['mask'], np.sqrt(np.maximum(row['w'].astype(float), 0)), 0)
                    residual = row['x'].astype(float) - pred
                    row['wmse'] = float(np.sum(weights * residual**2) / weights.sum())
                scores = np.array([r['wmse'] for r in rows])
                median = float(np.median(scores))
                order = np.argsort(np.abs(scores - median), kind='stable')
                used = set()
                for i in order:
                    row = rows[i]
                    identity = row['targetid'] if row['targetid'] != -1 else (row['source'], row['stream_row'])
                    if identity in used:
                        continue
                    used.add(identity)
                    selected.append(dict(row, pool_median_wmse=median, pool_count=len(rows)))
                    if len(used) == per_type:
                        break
                if len(used) < per_type:
                    raise ValueError(f'{kind}: insufficient distinct targets in sampled pool.')
    finally:
        model.train(was_training)
    return selected


def plot_examples(rows, wavelength=None, wavelength_label=None):
    """A 2-column paper figure; invalid bins break both spectrum and residual lines."""
    import matplotlib.pyplot as plt
    if not rows:
        raise ValueError('No examples.')
    length = len(rows[0]['x'])
    wave = np.arange(length) if wavelength is None else np.asarray(wavelength)
    if wave.shape != (length,) or not np.isfinite(wave).all() or np.any(np.diff(wave) <= 0):
        raise ValueError('Supply the exact increasing model wavelength grid.')
    if wavelength is not None and not wavelength_label:
        raise ValueError('State wavelength frame and units in wavelength_label.')
    names = dict(GALAXY='Galaxy', QSO='QSO', STAR='Star', TRANSIENT='Transient')
    with plt.rc_context({'font.size': 9, 'axes.labelsize': 9, 'pdf.fonttype': 42}):
        fig = plt.figure(figsize=(10, 3.0 * ((len(rows)+1)//2)), layout='constrained')
        grid = fig.add_gridspec((len(rows)+1)//2, 2)
        for i, row in enumerate(rows):
            inner = grid[i//2, i%2].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.04)
            ax = fig.add_subplot(inner[0])
            residual_ax = fig.add_subplot(inner[1], sharex=ax)
            valid = row['mask']
            observed = np.where(valid, row['x'], np.nan)
            reconstructed = np.where(valid, row['reconstruction'], np.nan)
            ax.plot(wave, observed, color='#454545', lw=0.7, label='Input')
            ax.plot(wave, reconstructed, color='#C45E16', lw=1.1, ls='--', label=r'VAE: decode($\mu$)')
            residual_ax.plot(wave, observed-reconstructed, color='#356A9A', lw=0.7)
            residual_ax.axhline(0, color='0.55', lw=0.6)
            ax.set_title(f"{names[row['kind']]}  |  TARGETID {row['targetid']}\nWMSE = {row['wmse']:.3g}", loc='left', fontsize=9)
            ax.set_ylabel('Preprocessed flux')
            residual_ax.set_ylabel('Residual')
            residual_ax.set_xlabel(wavelength_label or 'Model pixel index')
            ax.tick_params(labelbottom=False)
            if i == 0:
                ax.legend(frameon=False, fontsize=8)
            for axis in (ax, residual_ax):
                axis.spines[['top', 'right']].set_visible(False)
                axis.margins(x=0)
    return fig


def save_examples(fig, rows, output_dir, provenance):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_dir/'vae_example_reconstructions.pdf', bbox_inches='tight')
    fig.savefig(output_dir/'vae_example_reconstructions.png', dpi=300, bbox_inches='tight')
    arrays = {key: np.stack([r[key] for r in rows]) for key in ('x','w','mask','reconstruction')}
    np.savez_compressed(output_dir/'vae_example_reconstructions.npz', **arrays)
    metadata = [{k:v for k,v in r.items() if k not in arrays} for r in rows]
    with (output_dir/'selected_examples.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(metadata[0]))
        writer.writeheader()
        writer.writerows(metadata)
    (output_dir/'provenance.json').write_text(json.dumps(provenance, indent=2))
    (output_dir/'caption.txt').write_text(
        'Example VAE reconstructions of galaxy, QSO, stellar, and transient-sample spectra. '
        'The input preprocessed flux is shown in gray and the deterministic posterior-mean '
        'reconstruction in dashed orange; lower panels show input minus reconstruction. '
        'Masked pixels are omitted. Each example is selected nearest the median '
        'square-root-weighted MSE of its sampled candidate pool, not the full population. '
        'Selection parameters and source identifiers are recorded in the accompanying files. '
        'Transient-sample membership alone does not establish transient features at the observed epoch.\n')
    return output_dir


In [ ]:
# Configuration: replace paths only when the corresponding data live elsewhere.
EXAMPLE_DATASET = "/global/cfs/cdirs/desi/users/zixinq/data/dr2_catalog/loa_spectra_preprocessed"
EXAMPLE_TRANSIENT_NPZ = "/global/cfs/cdirs/desi/users/zixinq/data/dr2_catalog/transients_preprocessed_all.npz"
EXAMPLE_STAR_DATASET = None  # Verified stellar HDF5 directory, if stars are absent above.
MAX_BATCHES = 200           # First 200 batches; None scans the full background loader.
POOL_SIZE = 64
EXAMPLES_PER_TYPE = 1       # 1 gives four panels; 2 gives eight.
EXAMPLE_SEED = 42
EXAMPLE_WAVELENGTH = None   # Exact 1D model grid, not a guessed linspace.
EXAMPLE_WAVELENGTH_LABEL = None  # e.g. rest-frame wavelength plus units, after verification.
EXAMPLE_OUTPUT = Path("paper_reconstruction_examples")

example_loader = make_loader(EXAMPLE_DATASET, split="test", batch_size=256,
                             num_workers=0, shuffle=False, drop_last=False)
example_pools, background_scan = sample_examples(
    example_loader, ["GALAXY", "QSO", "STAR"], EXAMPLE_DATASET,
    pool_size=POOL_SIZE, max_batches=MAX_BATCHES, seed=EXAMPLE_SEED)
example_scans = [background_scan]
if EXAMPLE_STAR_DATASET is not None:
    example_star_loader = make_loader(EXAMPLE_STAR_DATASET, split="test", batch_size=256,
                                      num_workers=0, shuffle=False, drop_last=False)
    star_pool, star_scan = sample_examples(
        example_star_loader, ["STAR"], EXAMPLE_STAR_DATASET,
        pool_size=POOL_SIZE, max_batches=MAX_BATCHES, seed=EXAMPLE_SEED+1)
    example_pools["STAR"] = star_pool["STAR"]
    example_scans.append(star_scan)
transient_pool, transient_scan = sample_examples(
    transient_example_batches(EXAMPLE_TRANSIENT_NPZ), ["TRANSIENT"], EXAMPLE_TRANSIENT_NPZ,
    pool_size=POOL_SIZE, max_batches=None, seed=EXAMPLE_SEED+2)
example_pools.update(transient_pool)
example_scans.append(transient_scan)
print(json.dumps(example_scans, indent=2))
selected_examples = reconstruct_examples(model, example_pools, per_type=EXAMPLES_PER_TYPE)
example_fig = plot_examples(selected_examples, EXAMPLE_WAVELENGTH, EXAMPLE_WAVELENGTH_LABEL)
import matplotlib.pyplot as plt
plt.show()


In [ ]:
# Export after inspecting the displayed spectra and verifying the wavelength grid.
example_provenance = dict(
    checkpoint=str(ckpt_path), reconstruction="decode(encode(x).mu)",
    selection="Nearest sampled-pool median WMSE, distinct TARGETIDs within each class",
    wmse="sum(M*sqrt(max(W,0))*(X-Xhat)^2)/sum(M*sqrt(max(W,0)))",
    scans=example_scans, examples_per_type=EXAMPLES_PER_TYPE,
    background_selection="status=0, TARGETID hash bucket>=9000, no additional valid-fraction cut",
    row_identity="Background stream_row is the sequential loader offset; transient stream_row is NPZ source row. Saved NPZ arrays are in CSV row order.",
    training_exclusion="Not independently verified",
    wavelength_label=EXAMPLE_WAVELENGTH_LABEL or "Model pixel index",
    wavelength=None if EXAMPLE_WAVELENGTH is None else np.asarray(EXAMPLE_WAVELENGTH).tolist(),
)
saved_example_dir = save_examples(example_fig, selected_examples, EXAMPLE_OUTPUT, example_provenance)
print("Saved PDF, PNG, selected arrays, CSV, provenance, and draft caption to", saved_example_dir.resolve())


In [ ]:
# Run after your loading cell (expects mu_galaxy_test, mu_qso_test, mu_star_test, mu_tr, save_dir).
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
MAX_PER_CLASS = 5000  # balance classes so the big ones don't swamp the plot

class_latents = {
    "Galaxy": mu_galaxy_test,
    "QSO": mu_qso_test,
    "Star": mu_star_test,
    "Transient": mu_tr,  
}
colors = {"Galaxy": "tab:blue", "QSO": "tab:orange", "Star": "tab:green", "Transient": "tab:red"}
names = list(class_latents)

X_list, y_list = [], []
for name, mu in class_latents.items():
    mu = np.asarray(mu, dtype=np.float32).reshape(len(mu), -1)
    mu = mu[np.isfinite(mu).all(axis=1)]
    if len(mu) > MAX_PER_CLASS:
        mu = mu[rng.choice(len(mu), MAX_PER_CLASS, replace=False)]
    X_list.append(mu)
    y_list.append(np.full(len(mu), name))
    print(f"{name:10s}: {len(mu):6d} points, latent dim = {mu.shape[1]}")
X = np.vstack(X_list)
y = np.concatenate(y_list)
D = X.shape[1]
Xs = StandardScaler().fit_transform(X)

emb = {}
pca = PCA(n_components=2).fit(Xs)
emb[f"PCA ({pca.explained_variance_ratio_.sum():.0%} var)"] = pca.transform(Xs)

try:
    import umap
    emb["UMAP"] = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(Xs)
except ImportError:
    from sklearn.manifold import TSNE
    emb["t-SNE"] = TSNE(n_components=2, perplexity=30, init="pca", random_state=42).fit_transform(Xs)

# LDA = supervised projection that maximizes class separation (needs D >= 2)
if D >= 2:
    emb["LDA (supervised)"] = LinearDiscriminantAnalysis(n_components=2).fit_transform(Xs, y)


def scatter_by_class(ax, Z, title):
    for name in names:
        m = y == name
        tr = name == "Transient"
        ax.scatter(Z[m, 0], Z[m, 1], s=14 if tr else 3, alpha=0.9 if tr else 0.25,
                   c=colors[name], edgecolors="k" if tr else "none", linewidths=0.3,
                   label=f"{name} (n={m.sum()})", rasterized=True)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(markerscale=3, fontsize=8, loc="best")


fig, axes = plt.subplots(1, len(emb), figsize=(6 * len(emb), 5.5))
axes = np.atleast_1d(axes)
for ax, (title, Z) in zip(axes, emb.items()):
    scatter_by_class(ax, Z, title)
fig.tight_layout()
fig.savefig(os.path.join(save_dir, "latent_embeddings.png"), dpi=200)
plt.show()

F, _ = f_classif(X, y)
k = min(D, 5)
dims = np.argsort(F)[::-1][:k]
print("Most discriminative latent dims (by ANOVA F):", dims.tolist())

fig, axes = plt.subplots(k, k, figsize=(2.4 * k, 2.4 * k), squeeze=False)
for i, di in enumerate(dims):
    for j, dj in enumerate(dims):
        ax = axes[i, j]
        if j > i:
            ax.axis("off"); continue
        for name in names:
            m = y == name
            if i == j:
                ax.hist(X[m, di], bins=50, density=True, histtype="step", color=colors[name], lw=1.2)
            else:
                tr = name == "Transient"
                ax.scatter(X[m, dj], X[m, di], s=4 if tr else 1, alpha=0.8 if tr else 0.15,
                           c=colors[name], rasterized=True)
        if i == k - 1: ax.set_xlabel(f"z[{dj}]")
        if j == 0 and i > 0: ax.set_ylabel(f"z[{di}]")
handles = [plt.Line2D([], [], color=colors[n], lw=2, label=n) for n in names]
fig.legend(handles=handles, loc="upper right", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(save_dir, "latent_corner.png"), dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(max(8, 0.35 * D), 4))
offsets = np.linspace(-0.3, 0.3, len(names))
for off, name in zip(offsets, names):
    m = y == name
    ax.errorbar(np.arange(D) + off, X[m].mean(0), yerr=X[m].std(0), fmt="o", ms=3,
                capsize=2, color=colors[name], label=name)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("latent dimension"); ax.set_ylabel("μ (mean ± std)")
ax.legend(fontsize=8, ncol=len(names))
fig.tight_layout()
fig.savefig(os.path.join(save_dir, "latent_per_dim.png"), dpi=200)
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
MAX_PER_CLASS = 5000  # balance classes so the big ones don't swamp the plot

class_latents = {
    "Galaxy": mu_galaxy_test,
    "QSO": mu_qso_test,
    "Star": mu_star_test,
    "Transient": mu_tr,  
}
colors = {"Galaxy": "tab:blue", "QSO": "tab:orange", "Star": "tab:green", "Transient": "tab:red"}
names = list(class_latents)

X_list, y_list = [], []
for name, mu in class_latents.items():
    mu = np.asarray(mu, dtype=np.float32).reshape(len(mu), -1)
    mu = mu[np.isfinite(mu).all(axis=1)]
    if len(mu) > MAX_PER_CLASS:
        mu = mu[rng.choice(len(mu), MAX_PER_CLASS, replace=False)]
    X_list.append(mu)
    y_list.append(np.full(len(mu), name))
    print(f"{name:10s}: {len(mu):6d} points, latent dim = {mu.shape[1]}")
X = np.vstack(X_list)
y = np.concatenate(y_list)
D = X.shape[1]
Xs = StandardScaler().fit_transform(X)

In [ ]:
from sklearn.manifold import TSNE

def scatter_by_class(ax, Z, title):
    for name in names:
        m = y == name
        tr = name == "Transient"
        ax.scatter(Z[m, 0], Z[m, 1], s=14 if tr else 3, alpha=0.9 if tr else 0.25,
                   c=colors[name], edgecolors="k" if tr else "none", linewidths=0.3,
                   label=f"{name} (n={m.sum()})", rasterized=True)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(markerscale=3, fontsize=8, loc="best")

Z_tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=42).fit_transform(Xs)

fig, ax = plt.subplots(figsize=(7, 6))
scatter_by_class(ax, Z_tsne, "t-SNE")
fig.tight_layout()
fig.savefig(os.path.join(save_dir, "latent_tsne.png"), dpi=200)
plt.show()

In [ ]:
import umap
Z_umap = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(Xs)
ig, ax = plt.subplots(figsize=(7, 6))
scatter_by_class(ax, Z_umap, "UMAP")
fig.tight_layout()
fig.savefig(os.path.join(save_dir, "latent_umap.png"), dpi=200)
plt.show()